# NPC 3D CNN formal 4×5 — B+C开发 / A外部验证正式训练版

- 标签唯一来源：`$NPC_PROJECT_ROOT\NPC_3DCNN_mucositis_master_frozen_BCdev_Aexternal_v2.xlsx`中`exclude_reason`为空的489例`severe_mucositis`。
- 固定队列：`model_center B+C`为Development（309例），`model_center A`为External（180例）。
- NPZ继续使用既有v2影像数组；NPZ中的旧标签与旧cohort不读取、不比较。
- 结果目录：`BCdev_Aexternal_v2`，对应锁定的B+C开发/A外部分析配置。


> **最终489例分析配置**：使用最新冻结Master和v2固定4×5划分。Development=B+C=309（label 0/1=158/151），External=A=180完全锁定。训练输出目录为`formal_main_models_no_scalar_4x5_497_BCdev_Aexternal_v2`。

# NPC 3D CNN正式4×5无标量主模型训练

模型顺序：`M1、M2-Abl、M2、M3-Abl、M3、M4`。

M0（Clinical–DVH）保留为独立传统模型，不在本Notebook中训练。

程序支持断点续跑；直接运行下方唯一代码单元。

完成后先复核B+C Development的309例OOF结果；本Notebook不读取、不加载、不评价Center A External。

In [ ]:
# -*- coding: utf-8 -*-
r"""
NPC重度急性口腔黏膜炎3D dose-map项目
正式4次重复×5折无标量主模型训练、OOF汇总与关键配对比较
================================================================================

脚本名称
--------
NPC_3DCNN_formal_4x5_main_models_no_scalar_v1.py

最终模型命名
------------
M0: Clinical–DVH model
    独立传统模型管线，本脚本不训练M0。

M1: Dose-only model
    dose

M2-Abl: M2 without oral masked-dose
    dose + oral

M2: Oral masked-dose model
    dose + oral + dose_oral

M3-Abl: M3 without oral/GTV masked-dose
    dose + oral + gtv

M3: Oral/GTV masked-dose model
    dose + oral + gtv + dose_oral + dose_gtv

M4: CT-augmented M3
    ct + dose + oral + gtv + dose_oral + dose_gtv

核心比较
--------
1. M2-Abl vs M2：
   评价oral masked-dose通道的增量价值。

2. M3-Abl vs M3：
   评价oral/GTV masked-dose通道的增量价值。

3. M3 vs M4：
   评价planning CT的增量价值。

重要原则
--------
1. 本脚本训练M1、M2-Abl、M2、M3-Abl、M3、M4；
   全部不使用任何标量分支。
2. M0（Clinical–DVH）保留为独立传统基准，后续使用单独代码建模。
3. dose_oral和dose_gtv在Dataset中实时计算：
       dose_oral = dose × (oral > 0.5)
       dose_gtv  = dose × (gtv  > 0.5)
4. 直接读取冻结的4 repeats × 5 folds及每个fold的固定inner split。
5. 每个model-fold均执行：
       Stage A：inner_train训练，inner_validation选择epoch；
       Stage B：完整outer_train从头训练selected_epoch，
                outer_validation仅用于正式OOF评价。
6. 同一fold内所有CNN模型使用fold_config.json中相同training_seed。
7. Center A External不读取、不加载、不预测、不评价。
8. 程序支持断点续跑：
   已完整PASS的model-fold自动跳过；
   中断留下的不完整任务目录自动删除后重跑。
9. 完成120个model-fold任务后自动生成：
   - 20-fold汇总；
   - 每个repeat完整309例OOF；
   - 每例4次OOF预测及患者级平均概率；
   - M2-Abl vs M2、M3-Abl vs M3、M3 vs M4配对bootstrap比较；
   - 完整审计文件。
10. 不能把1236条重复OOF预测视为1236个独立病例。
    最终患者级汇总以309例、每例4次OOF概率平均为基础。

正式数据
--------
NPZ：
$NPC_PROJECT_ROOT\
preprocessed_497_2x2x3_patch80x112x64_v2\npz

固定划分：
$NPC_PROJECT_ROOT\
fixed_splits_repeated_5fold_4repeats_497_BCdev_Aexternal_v2

正式训练范围
------------
4 repeats × 5 folds × 6 CNN models
= 120 model-fold任务
= 120次Stage A + 120次Stage B

训练管线
--------
沿用已通过single-fold工程验证的稳定设置：
- Lightweight 3D ResNet-10
- GroupNorm
- AdamW
- warm-up + cosine learning-rate schedule
- Stage A使用3-epoch平滑inner loss选择epoch
- CUDA优先BF16，不支持时回退FP16
- 概率和评价指标使用FP32
- Center B与Center C分别报告
- External始终隔离
"""

# 必须放在import torch之前
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

from pathlib import Path
from datetime import datetime
import gc
import hashlib
import json
import math
import random
import shutil
import sys
import time
import traceback
import warnings

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    auc,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    precision_recall_curve,
    roc_auc_score,
)

import matplotlib.pyplot as plt


warnings.filterwarnings("ignore", category=RuntimeWarning)


# =============================================================================
# 1. 路径与正式运行控制
# =============================================================================

def require_env_path(name: str) -> Path:
    """Return a required absolute path from an environment variable."""
    value = os.environ.get(name)
    if not value:
        raise RuntimeError(
            f"Set {name} to an absolute path before running this notebook."
        )
    path = Path(value).expanduser()
    if not path.is_absolute():
        raise RuntimeError(f"{name} must be an absolute path: {value!r}")
    return path.resolve()

PROJECT_DIR = require_env_path("NPC_PROJECT_ROOT")

# 唯一训练标签来源：正式master Excel。
MASTER_XLSX = (
    PROJECT_DIR
    / "NPC_3DCNN_mucositis_master_frozen_BCdev_Aexternal_v2.xlsx"
)

EXPECTED_MASTER_SHA256 = (
    "31a72c9d3e1b21d69b8b227eee5d21b9"
    "c2129624401f5d3176c0b1770e148c93"
)
STRICT_MASTER_SHA256 = True

NPZ_DIR = (
    PROJECT_DIR
    / "preprocessed_497_2x2x3_patch80x112x64_v2"
    / "npz"
)

SPLIT_ROOT = (
    PROJECT_DIR
    / (
        "fixed_splits_repeated_5fold_"
        "4repeats_497_BCdev_Aexternal_v2"
    )
)

SPLITS_LOCKED_JSON = (
    SPLIT_ROOT
    / "SPLITS_LOCKED.json"
)

OUTPUT_VERSION = "BCdev_Aexternal_v2"

FORMAL_OUTPUT_ROOT = (
    PROJECT_DIR
    / (
        "formal_main_models_no_scalar_"
        "4x5_497_"
        + OUTPUT_VERSION
    )
)

AGGREGATE_DIR = (
    FORMAL_OUTPUT_ROOT
    / "aggregate"
)

# 正式运行范围。默认完整4×5。
REPEATS_TO_RUN = [
    1,
    2,
    3,
    4,
]

FOLDS_TO_RUN = [
    1,
    2,
    3,
    4,
    5,
]

# 断点续跑设置。
ALLOW_RESUME = True
RERUN_INCOMPLETE_TASKS = True
STOP_ON_TASK_ERROR = True

# 完成全部任务后进行患者级配对分层bootstrap。
PAIRED_BOOTSTRAP_ITERATIONS = 2000
PAIRED_BOOTSTRAP_SEED = 60260701

REQUIRE_CUDA = True

BUILD_NOTE = (
    "Formal 4x5 repeated stratified CV; "
    "Development=model_center B+C (n=309), External=model_center A (n=180); "
    "no scalar branch; locked v2 NPZ image arrays and regenerated "
    "BCdev_Aexternal_v2 outer/inner split CSVs; "
    "labels loaded from included cases in the frozen Master; "
    "Master legacy cohort is ignored; split CSV labels are remapped from Master; "
    "NPZ labels/cohort are neither read nor checked; "
    "conservative Stage-A early stopping: min epoch 40, patience 25"
)

REPEAT_NUMBER = None
FOLD_NUMBER = None
FOLD_DIR = None
OUTPUT_ROOT = None

GLOBAL_SEED = None
INNER_SPLIT_SEED = None
OUTER_SPLIT_SEED = None


# =============================================================================
# 2. 最终模型输入与命名：全部无标量分支
# =============================================================================

MODEL_CHANNELS = {
    "M1_Dose_Only": [
        "dose",
    ],

    "M2_Abl": [
        "dose",
        "oral",
    ],

    "M2_Oral_MaskedDose": [
        "dose",
        "oral",
        "dose_oral",
    ],

    "M3_Abl": [
        "dose",
        "oral",
        "gtv",
    ],

    "M3_Oral_GTV_MaskedDose": [
        "dose",
        "oral",
        "gtv",
        "dose_oral",
        "dose_gtv",
    ],

    "M4_CT_Augmented_M3": [
        "ct",
        "dose",
        "oral",
        "gtv",
        "dose_oral",
        "dose_gtv",
    ],
}

MODEL_DISPLAY_NAMES = {
    "M1_Dose_Only": (
        "M1"
    ),

    "M2_Abl": (
        "M2-Abl"
    ),

    "M2_Oral_MaskedDose": (
        "M2"
    ),

    "M3_Abl": (
        "M3-Abl"
    ),

    "M3_Oral_GTV_MaskedDose": (
        "M3"
    ),

    "M4_CT_Augmented_M3": (
        "M4"
    ),
}

MODEL_FULL_NAMES = {
    "M1_Dose_Only": (
        "M1: Dose-only model"
    ),

    "M2_Abl": (
        "M2-Abl: M2 without oral masked-dose"
    ),

    "M2_Oral_MaskedDose": (
        "M2: Oral masked-dose model"
    ),

    "M3_Abl": (
        "M3-Abl: M3 without oral/GTV masked-dose"
    ),

    "M3_Oral_GTV_MaskedDose": (
        "M3: Oral/GTV masked-dose model"
    ),

    "M4_CT_Augmented_M3": (
        "M4: CT-augmented M3"
    ),
}

MODEL_PURPOSES = {
    "M1_Dose_Only": (
        "Dose-only deep-learning baseline"
    ),

    "M2_Abl": (
        "Ablated M2 without dose_oral"
    ),

    "M2_Oral_MaskedDose": (
        "Oral masked-dose model; compare with M2-Abl"
    ),

    "M3_Abl": (
        "Ablated M3 without dose_oral and dose_gtv"
    ),

    "M3_Oral_GTV_MaskedDose": (
        "Primary oral/GTV spatial masked-dose model; "
        "compare with M3-Abl"
    ),

    "M4_CT_Augmented_M3": (
        "CT-augmented M3; compare with M3"
    ),
}

MODELS_TO_RUN = [
    "M1_Dose_Only",
    "M2_Abl",
    "M2_Oral_MaskedDose",
    "M3_Abl",
    "M3_Oral_GTV_MaskedDose",
    "M4_CT_Augmented_M3",
]

PAIRWISE_COMPARISONS = [
    {
        "comparison": (
            "M2-Abl vs M2"
        ),
        "reference_model": (
            "M2_Abl"
        ),
        "candidate_model": (
            "M2_Oral_MaskedDose"
        ),
        "question": (
            "Incremental value of oral masked-dose"
        ),
    },

    {
        "comparison": (
            "M3-Abl vs M3"
        ),
        "reference_model": (
            "M3_Abl"
        ),
        "candidate_model": (
            "M3_Oral_GTV_MaskedDose"
        ),
        "question": (
            "Incremental value of oral/GTV masked-dose"
        ),
    },

    {
        "comparison": (
            "M3 vs M4"
        ),
        "reference_model": (
            "M3_Oral_GTV_MaskedDose"
        ),
        "candidate_model": (
            "M4_CT_Augmented_M3"
        ),
        "question": (
            "Incremental value of planning CT"
        ),
    },
]

DERIVED_MASKED_DOSE_CHANNELS = {
    "dose_oral",
    "dose_gtv",
}

ALLOWED_IMAGE_CHANNELS = {
    "ct",
    "dose",
    "oral",
    "gtv",
    "dose_oral",
    "dose_gtv",
}

FULL_MODEL_DESIGN_ROWS = [
    {
        "display_name": (
            "M0"
        ),
        "full_name": (
            "M0: Clinical-DVH model"
        ),
        "input_channels_or_features": (
            "Clinical + DVH"
        ),
        "trained_in_this_script": (
            False
        ),
        "role": (
            "Traditional baseline; separate pipeline"
        ),
    },

    *[
        {
            "display_name": (
                MODEL_DISPLAY_NAMES[
                    model_name
                ]
            ),
            "full_name": (
                MODEL_FULL_NAMES[
                    model_name
                ]
            ),
            "input_channels_or_features": (
                " + ".join(
                    MODEL_CHANNELS[
                        model_name
                    ]
                )
            ),
            "trained_in_this_script": (
                True
            ),
            "role": (
                MODEL_PURPOSES[
                    model_name
                ]
            ),
        }
        for model_name
        in MODELS_TO_RUN
    ],
]


# =============================================================================
# 3. 固定训练参数
# =============================================================================

EXPECTED_SHAPE_ZYX = (
    64,
    112,
    80,
)

INNER_VALIDATION_FRACTION = 0.20

BASE_CHANNELS = 16
DROPOUT = 0.20

BATCH_SIZE = 4
NUM_WORKERS = 0

MAX_EPOCHS = 120
MIN_CHECKPOINT_EPOCH = 10
MIN_EARLY_STOPPING_EPOCH = 40
EARLY_STOPPING_PATIENCE = 25
EARLY_STOPPING_MIN_DELTA = 1e-4

# 以连续3个epoch的inner loss平均值作为checkpoint选择分数
SELECTION_SMOOTHING_WINDOW = 3

BASE_LR = 1e-3
MIN_LR = 5e-5
WARMUP_EPOCHS = 5

WEIGHT_DECAY = 1e-4
MAX_GRAD_NORM = 5.0

USE_AMP = True
# auto：CUDA支持时优先BF16，否则回退FP16；所有指标和概率均使用FP32。
AMP_PRECISION_MODE = "auto"
ENABLE_AUGMENTATION = False

# 0.5仅用于描述分类指标，不用于模型选择
DESCRIPTIVE_THRESHOLD = 0.5


# =============================================================================
# 4. 正式输出文件
# =============================================================================

FORMAL_RUN_CONFIG_JSON = (
    FORMAL_OUTPUT_ROOT
    / "formal_4x5_run_config.json"
)

FORMAL_PROGRESS_CSV = (
    FORMAL_OUTPUT_ROOT
    / "formal_4x5_task_progress.csv"
)

FORMAL_RUN_LOG_TXT = (
    FORMAL_OUTPUT_ROOT
    / "formal_4x5_run_log.txt"
)

FORMAL_COMPLETED_JSON = (
    FORMAL_OUTPUT_ROOT
    / "FORMAL_4X5_COMPLETED.json"
)

ALL_TASKS_SUMMARY_CSV = (
    AGGREGATE_DIR
    / "formal_4x5_all_tasks_summary.csv"
)

FOLD_METRICS_CSV = (
    AGGREGATE_DIR
    / "formal_4x5_fold_metrics.csv"
)

REPEAT_METRICS_CSV = (
    AGGREGATE_DIR
    / "formal_4x5_repeat_metrics.csv"
)

PATIENT_AVERAGED_METRICS_CSV = (
    AGGREGATE_DIR
    / "formal_4x5_patient_averaged_metrics.csv"
)

MODEL_SUMMARY_CSV = (
    AGGREGATE_DIR
    / "formal_4x5_model_summary.csv"
)

PAIRED_COMPARISONS_CSV = (
    AGGREGATE_DIR
    / "formal_4x5_paired_comparisons.csv"
)

FORMAL_SUMMARY_XLSX = (
    AGGREGATE_DIR
    / "formal_4x5_summary.xlsx"
)

AGGREGATE_AUDIT_JSON = (
    AGGREGATE_DIR
    / "formal_4x5_aggregate_audit.json"
)


# =============================================================================
# 5. 通用函数
# =============================================================================

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

    try:
        torch.use_deterministic_algorithms(
            True,
            warn_only=True,
        )
    except Exception:
        pass


def save_json(
    payload,
    output_path: Path,
) -> None:
    def convert(value):
        if isinstance(value, Path):
            return str(value)

        if isinstance(value, dict):
            return {
                str(key): convert(item)
                for key, item in value.items()
            }

        if isinstance(value, (list, tuple)):
            return [
                convert(item)
                for item in value
            ]

        if isinstance(value, np.generic):
            return value.item()

        return value

    output_path.write_text(
        json.dumps(
            convert(payload),
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )


def sha256_file(
    file_path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:
    digest = hashlib.sha256()

    with open(file_path, "rb") as file:
        while True:
            block = file.read(
                chunk_size
            )

            if not block:
                break

            digest.update(
                block
            )

    return digest.hexdigest()


def safe_auc(
    y_true,
    probabilities,
) -> float:
    try:
        return float(
            roc_auc_score(
                y_true,
                probabilities,
            )
        )
    except Exception:
        return np.nan


def safe_pr_auc(
    y_true,
    probabilities,
) -> float:
    """Trapezoidal PR-AUC with recall on the x-axis (manuscript definition)."""
    try:
        precision, recall, _ = precision_recall_curve(
            y_true,
            probabilities,
        )
        return float(auc(recall, precision))
    except Exception:
        return np.nan


def calculate_binary_metrics(
    y_true,
    probabilities,
    threshold: float = 0.5,
) -> dict:
    y_true = np.asarray(
        y_true,
        dtype=int,
    )

    probabilities = np.asarray(
        probabilities,
        dtype=float,
    )

    predictions = (
        probabilities >= threshold
    ).astype(int)

    cm = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1],
    )

    tn, fp, fn, tp = [
        int(value)
        for value
        in cm.ravel()
    ]

    sensitivity = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else np.nan
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    return {
        "n": int(
            len(y_true)
        ),

        "prevalence": float(
            y_true.mean()
        ),

        "roc_auc": safe_auc(
            y_true,
            probabilities,
        ),

        "pr_auc": safe_pr_auc(
            y_true,
            probabilities,
        ),

        "brier": float(
            brier_score_loss(
                y_true,
                probabilities,
            )
        ),

        "threshold_descriptive_only": float(
            threshold
        ),

        "accuracy": float(
            accuracy_score(
                y_true,
                predictions,
            )
        ),

        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                predictions,
            )
        ),

        "sensitivity": float(
            sensitivity
        ),

        "specificity": float(
            specificity
        ),

        "precision": float(
            precision_score(
                y_true,
                predictions,
                zero_division=0,
            )
        ),

        "f1": float(
            f1_score(
                y_true,
                predictions,
                zero_division=0,
            )
        ),

        "probability_min": float(
            probabilities.min()
        ),

        "probability_max": float(
            probabilities.max()
        ),

        "probability_mean": float(
            probabilities.mean()
        ),

        "probability_std": float(
            probabilities.std(
                ddof=1
            )
        ),

        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
    }


def count_trainable_parameters(
    model: nn.Module,
) -> int:
    return int(
        sum(
            parameter.numel()
            for parameter
            in model.parameters()
            if parameter.requires_grad
        )
    )


def learning_rate_for_epoch(
    epoch: int,
) -> float:
    """
    固定的绝对epoch学习率计划。

    Stage A与Stage B均使用相同函数。
    Stage B即使只训练selected_epoch，也不会将余弦周期压缩。
    """
    if epoch <= WARMUP_EPOCHS:
        if WARMUP_EPOCHS == 1:
            warmup_fraction = 1.0

        else:
            warmup_fraction = (
                epoch - 1
            ) / (
                WARMUP_EPOCHS - 1
            )

        start_ratio = 0.20

        ratio = (
            start_ratio
            +
            (
                1.0 - start_ratio
            )
            * warmup_fraction
        )

        return float(
            BASE_LR
            * ratio
        )

    progress = (
        epoch - WARMUP_EPOCHS
    ) / (
        MAX_EPOCHS - WARMUP_EPOCHS
    )

    progress = min(
        max(
            progress,
            0.0,
        ),
        1.0,
    )

    cosine_value = 0.5 * (
        1.0
        +
        math.cos(
            math.pi
            * progress
        )
    )

    return float(
        MIN_LR
        +
        (
            BASE_LR
            - MIN_LR
        )
        * cosine_value
    )


def set_optimizer_lr(
    optimizer,
    learning_rate: float,
) -> None:
    for parameter_group in (
        optimizer.param_groups
    ):
        parameter_group[
            "lr"
        ] = float(
            learning_rate
        )


def resolve_amp_settings(
    device: torch.device,
) -> dict:
    """
    自动选择训练精度。

    优先级：
    1. CUDA + BF16支持：BF16 autocast，不使用GradScaler；
    2. CUDA但不支持BF16：FP16 autocast，使用GradScaler；
    3. USE_AMP=False：全FP32。
    """
    if (
        not USE_AMP
        or
        device.type != "cuda"
    ):
        return {
            "enabled": False,
            "dtype": torch.float32,
            "mode": "FP32",
            "scaler_enabled": False,
        }

    bf16_supported = False

    if hasattr(
        torch.cuda,
        "is_bf16_supported",
    ):
        try:
            bf16_supported = bool(
                torch.cuda.is_bf16_supported()
            )
        except Exception:
            bf16_supported = False

    if (
        AMP_PRECISION_MODE
        in {
            "auto",
            "bf16",
        }
        and
        bf16_supported
    ):
        return {
            "enabled": True,
            "dtype": torch.bfloat16,
            "mode": "BF16",
            "scaler_enabled": False,
        }

    return {
        "enabled": True,
        "dtype": torch.float16,
        "mode": "FP16",
        "scaler_enabled": True,
    }


# =============================================================================
# 6. 读取和审计固定划分
# =============================================================================

MASTER_LABEL_LOOKUP = None
MASTER_LABEL_AUDIT = None


def load_master_label_source() -> dict:
    """
    读取本次实验Frozen Master并建立patient_id -> severe_mucositis映射。
    仅使用exclude_reason为空的489例；Master旧cohort字段不参与本次队列定义。
    """
    global MASTER_LABEL_LOOKUP
    global MASTER_LABEL_AUDIT

    if not MASTER_XLSX.exists():
        raise FileNotFoundError(
            "正式master Excel不存在：\n"
            f"{MASTER_XLSX}"
        )

    current_master_sha256 = sha256_file(
        MASTER_XLSX
    )

    if (
        STRICT_MASTER_SHA256
        and
        current_master_sha256
        != EXPECTED_MASTER_SHA256
    ):
        raise RuntimeError(
            "Frozen Master SHA256不匹配，程序停止。\n"
            f"当前SHA256：{current_master_sha256}\n"
            f"预期SHA256：{EXPECTED_MASTER_SHA256}\n"
            f"文件：{MASTER_XLSX}"
        )

    dataframe = pd.read_excel(
        MASTER_XLSX,
        sheet_name=0,
        dtype=object,
        engine="openpyxl",
    )

    dataframe.columns = [
        str(column).strip()
        for column in dataframe.columns
    ]

    required_master_columns = {
        "patient_id",
        "model_center",
        "exclude_reason",
        "severe_mucositis",
    }

    missing_columns = (
        required_master_columns
        - set(dataframe.columns)
    )

    if missing_columns:
        raise KeyError(
            "正式master缺少字段："
            f"{sorted(missing_columns)}"
        )

    dataframe = dataframe.loc[
        dataframe["patient_id"].notna()
    ].copy()

    dataframe["patient_id"] = pd.to_numeric(
        dataframe["patient_id"],
        errors="raise",
    ).astype(int)

    if dataframe["patient_id"].duplicated().any():
        raise RuntimeError(
            "正式master存在重复patient_id"
        )

    if len(dataframe) != 540:
        raise RuntimeError(
            f"Master总病例数错误：{len(dataframe)} != 540"
        )

    excluded_text = (
        dataframe["exclude_reason"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    included = dataframe.loc[
        excluded_text.eq("")
    ].copy()

    excluded = dataframe.loc[
        ~excluded_text.eq("")
    ].copy()

    if len(excluded) != 51:
        raise RuntimeError(
            f"Master排除病例数错误：{len(excluded)} != 51"
        )

    if len(included) != 489:
        raise RuntimeError(
            f"Master正式纳入病例数错误：{len(included)} != 489"
        )

    included["model_center"] = (
        included["model_center"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    numeric_labels = pd.to_numeric(
        included["severe_mucositis"],
        errors="coerce",
    )

    if not numeric_labels.isin([0, 1]).all():
        invalid_rows = included.loc[
            ~numeric_labels.isin([0, 1]),
            [
                "patient_id",
                "model_center",
                "severe_mucositis",
            ],
        ].head(20)

        raise RuntimeError(
            "正式纳入病例存在缺失或非0/1标签：\n"
            + invalid_rows.to_string(index=False)
        )

    included["severe_mucositis"] = (
        numeric_labels.astype(int)
    )

    center_counts = (
        included["model_center"]
        .value_counts()
        .to_dict()
    )

    expected_center_counts = {
        "A": 180,
        "B": 81,
        "C": 228,
    }

    if center_counts != expected_center_counts:
        raise RuntimeError(
            f"Master中心数量错误：{center_counts}"
        )

    label_counts = (
        included["severe_mucositis"]
        .value_counts()
        .sort_index()
        .to_dict()
    )

    if label_counts != {
        0: 246,
        1: 243,
    }:
        raise RuntimeError(
            f"Master标签数量错误：{label_counts}"
        )

    center_label_counts = {
        (str(center), int(label)): int(count)
        for (center, label), count
        in included.groupby(
            [
                "model_center",
                "severe_mucositis",
            ]
        ).size().items()
    }

    expected_center_label_counts = {
        ("A", 0): 88,
        ("A", 1): 92,
        ("B", 0): 42,
        ("B", 1): 39,
        ("C", 0): 116,
        ("C", 1): 112,
    }

    if center_label_counts != expected_center_label_counts:
        raise RuntimeError(
            "Master中心×标签数量错误："
            f"{center_label_counts}"
        )

    MASTER_LABEL_LOOKUP = dict(
        zip(
            included["patient_id"].astype(int),
            included["severe_mucositis"].astype(int),
        )
    )

    MASTER_LABEL_AUDIT = {
        "master_filename": MASTER_XLSX.name,
        "master_path": str(MASTER_XLSX),
        "master_sha256": current_master_sha256,
        "master_rows_with_patient_id": int(len(dataframe)),
        "master_excluded_cases": int(len(excluded)),
        "master_rows_with_valid_binary_label": 489,
        "master_total_cases": 489,
        "master_positive_cases": 243,
        "master_negative_cases": 246,
        "master_center_counts": expected_center_counts,
        "development_definition": "model_center B+C",
        "development_cases": 309,
        "development_label0": 158,
        "development_label1": 151,
        "external_definition": "model_center A",
        "external_cases": 180,
        "external_label0": 88,
        "external_label1": 92,
        "master_legacy_cohort_used": False,
        "label_source": (
            "NPC_3DCNN_mucositis_master_frozen_BCdev_Aexternal_v2.xlsx"
        ),
        "split_csv_label_read": False,
        "split_csv_label_compared": False,
        "npz_label_read": False,
        "npz_label_checked": False,
        "npz_cohort_used": False,
    }

    return MASTER_LABEL_AUDIT


REQUIRED_COLUMNS = {
    "patient_id",
    "model_center",
    "cohort",
    "severe_mucositis",
    "stratum",
    "npz_path",
}


def read_split_csv(
    path: Path,
    role_name: str,
) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(
            f"{role_name}不存在：\n"
            f"{path}"
        )

    dataframe = pd.read_csv(
        path,
        encoding="utf-8-sig",
    )

    required_split_columns = {
        "patient_id",
        "model_center",
        "cohort",
        "npz_path",
    }

    missing_columns = (
        required_split_columns
        - set(dataframe.columns)
    )

    if missing_columns:
        raise KeyError(
            f"{role_name}缺少字段："
            f"{sorted(missing_columns)}"
        )

    dataframe["patient_id"] = pd.to_numeric(
        dataframe["patient_id"],
        errors="raise",
    ).astype(int)

    dataframe["model_center"] = (
        dataframe["model_center"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    dataframe["cohort"] = (
        dataframe["cohort"]
        .astype(str)
        .str.strip()
    )

    dataframe["npz_path"] = (
        dataframe["npz_path"]
        .astype(str)
        .map(Path)
    )

    if dataframe["patient_id"].duplicated().any():
        raise RuntimeError(
            f"{role_name}存在重复patient_id"
        )

    if MASTER_LABEL_LOOKUP is None:
        load_master_label_source()

    missing_master_labels = sorted(
        patient_id
        for patient_id
        in dataframe["patient_id"].astype(int)
        if patient_id not in MASTER_LABEL_LOOKUP
    )

    if missing_master_labels:
        raise RuntimeError(
            f"{role_name}存在master缺失标签的病例："
            f"{missing_master_labels[:20]}"
        )

    # 固定split CSV只用于既有病例分配及路径；其中的标签和stratum不读取、不比较。
    # NPZ内部label/severe_mucositis不读取、不比较。
    dataframe["severe_mucositis"] = (
        dataframe["patient_id"]
        .map(MASTER_LABEL_LOOKUP)
        .astype(int)
    )

    dataframe["stratum"] = (
        dataframe["model_center"]
        + "_label"
        + dataframe["severe_mucositis"].astype(str)
    )

    if not set(
        dataframe["severe_mucositis"].unique()
    ).issubset({0, 1}):
        raise RuntimeError(
            f"{role_name}从master读取的标签不是0/1"
        )

    if set(
        dataframe["cohort"].unique()
    ) != {"Development"}:
        raise RuntimeError(
            f"{role_name}必须仅包含Development"
        )

    if not set(
        dataframe["model_center"].unique()
    ).issubset({"B", "C"}):
        raise RuntimeError(
            f"{role_name}出现非B/C开发中心"
        )

    missing_npz = [
        str(npz_path)
        for npz_path
        in dataframe["npz_path"]
        if not Path(npz_path).exists()
    ]

    if missing_npz:
        raise FileNotFoundError(
            f"{role_name}存在缺失NPZ：\n"
            + "\n".join(missing_npz[:20])
        )

    wrong_npz_root = []
    expected_npz_root = NPZ_DIR.resolve()

    for npz_path in dataframe["npz_path"]:
        npz_path = Path(npz_path)

        try:
            actual_parent = npz_path.resolve().parent
        except Exception:
            actual_parent = npz_path.parent

        if actual_parent != expected_npz_root:
            wrong_npz_root.append(
                str(npz_path)
            )

    if wrong_npz_root:
        raise RuntimeError(
            f"{role_name}存在未指向v2 NPZ目录的路径：\n"
            + "\n".join(wrong_npz_root[:20])
        )

    return (
        dataframe
        .sort_values("patient_id")
        .reset_index(drop=True)
    )


def audit_locked_split_relationships(
    outer_train_df: pd.DataFrame,
    outer_validation_df: pd.DataFrame,
    inner_train_df: pd.DataFrame,
    inner_validation_df: pd.DataFrame,
) -> dict:
    outer_train_ids = set(
        outer_train_df[
            "patient_id"
        ]
    )

    outer_validation_ids = set(
        outer_validation_df[
            "patient_id"
        ]
    )

    inner_train_ids = set(
        inner_train_df[
            "patient_id"
        ]
    )

    inner_validation_ids = set(
        inner_validation_df[
            "patient_id"
        ]
    )

    if (
        outer_train_ids
        & outer_validation_ids
    ):
        raise RuntimeError(
            "outer_train与outer_validation重叠"
        )

    if len(
        outer_train_ids
        | outer_validation_ids
    ) != 309:
        raise RuntimeError(
            "outer train/validation未完整覆盖309例Development"
        )

    if (
        inner_train_ids
        & inner_validation_ids
    ):
        raise RuntimeError(
            "inner_train与inner_validation重叠"
        )

    if (
        inner_train_ids
        | inner_validation_ids
    ) != outer_train_ids:
        raise RuntimeError(
            "固定inner split未完整覆盖outer_train"
        )

    if (
        inner_train_ids
        | inner_validation_ids
    ) & outer_validation_ids:
        raise RuntimeError(
            "固定inner split混入outer_validation"
        )

    if len(
        outer_train_df
    ) not in {
        247,
        248,
    }:
        raise RuntimeError(
            "outer_train数量异常："
            f"{len(outer_train_df)}"
        )

    if len(
        outer_validation_df
    ) not in {
        61,
        62,
    }:
        raise RuntimeError(
            "outer_validation数量异常："
            f"{len(outer_validation_df)}"
        )

    if len(
        inner_validation_df
    ) != 50:
        raise RuntimeError(
            "inner_validation数量异常："
            f"{len(inner_validation_df)}"
        )

    # 同一病例出现在不同角色时，核心元数据必须完全一致。
    reference = pd.concat(
        [
            outer_train_df,
            outer_validation_df,
        ],
        ignore_index=True,
    )[
        [
            "patient_id",
            "model_center",
            "cohort",
            "severe_mucositis",
            "stratum",
            "npz_path",
        ]
    ].copy()

    reference = reference.set_index(
        "patient_id"
    )

    for role_name, dataframe in {
        "inner_train": (
            inner_train_df
        ),
        "inner_validation": (
            inner_validation_df
        ),
    }.items():
        for _, row in dataframe.iterrows():
            patient_id = int(
                row[
                    "patient_id"
                ]
            )

            expected = reference.loc[
                patient_id
            ]

            for column in [
                "model_center",
                "cohort",
                "severe_mucositis",
                "stratum",
                "npz_path",
            ]:
                if str(
                    row[
                        column
                    ]
                ) != str(
                    expected[
                        column
                    ]
                ):
                    raise RuntimeError(
                        f"{role_name}病例{patient_id}"
                        f"的{column}不一致"
                    )

    expected_strata = {
        "B_label0",
        "B_label1",
        "C_label0",
        "C_label1",
    }

    for role_name, dataframe in {
        "outer_train": (
            outer_train_df
        ),
        "outer_validation": (
            outer_validation_df
        ),
        "inner_train": (
            inner_train_df
        ),
        "inner_validation": (
            inner_validation_df
        ),
    }.items():
        if set(
            dataframe[
                "stratum"
            ].unique()
        ) != expected_strata:
            raise RuntimeError(
                f"{role_name}没有覆盖全部4个strata"
            )

    return {
        "outer_train_n": (
            len(
                outer_train_df
            )
        ),
        "outer_validation_n": (
            len(
                outer_validation_df
            )
        ),
        "inner_train_n": (
            len(
                inner_train_df
            )
        ),
        "inner_validation_n": (
            len(
                inner_validation_df
            )
        ),
        "outer_train_strata": (
            outer_train_df[
                "stratum"
            ]
            .value_counts()
            .sort_index()
            .astype(int)
            .to_dict()
        ),
        "outer_validation_strata": (
            outer_validation_df[
                "stratum"
            ]
            .value_counts()
            .sort_index()
            .astype(int)
            .to_dict()
        ),
        "inner_train_strata": (
            inner_train_df[
                "stratum"
            ]
            .value_counts()
            .sort_index()
            .astype(int)
            .to_dict()
        ),
        "inner_validation_strata": (
            inner_validation_df[
                "stratum"
            ]
            .value_counts()
            .sort_index()
            .astype(int)
            .to_dict()
        ),
        "status": (
            "PASS"
        ),
    }


# =============================================================================
# 7. Dataset：实时构建dose_oral和dose_gtv
# =============================================================================

class NPCNPZDataset(Dataset):
    def __init__(
        self,
        dataframe: pd.DataFrame,
        channels,
        training: bool,
    ):
        self.dataframe = (
            dataframe
            .copy()
            .reset_index(
                drop=True
            )
        )

        self.channels = list(
            channels
        )

        self.training = bool(
            training
        )

        unknown_channels = (
            set(
                self.channels
            )
            - ALLOWED_IMAGE_CHANNELS
        )

        if unknown_channels:
            raise ValueError(
                "未知图像通道："
                f"{sorted(unknown_channels)}"
            )

        if (
            "dose_oral"
            in self.channels
            and
            not {
                "dose",
                "oral",
            }.issubset(
                set(
                    self.channels
                )
            )
        ):
            raise ValueError(
                "使用dose_oral时，"
                "模型输入必须同时包含dose和oral"
            )

        if (
            "dose_gtv"
            in self.channels
            and
            not {
                "dose",
                "gtv",
            }.issubset(
                set(
                    self.channels
                )
            )
        ):
            raise ValueError(
                "使用dose_gtv时，"
                "模型输入必须同时包含dose和gtv"
            )

    def __len__(
        self,
    ):
        return len(
            self.dataframe
        )

    def __getitem__(
        self,
        index,
    ):
        row = self.dataframe.iloc[
            index
        ]

        patient_id = int(
            row[
                "patient_id"
            ]
        )

        label = int(
            row[
                "severe_mucositis"
            ]
        )

        npz_path = Path(
            row[
                "npz_path"
            ]
        )

        required_base_channels = {
            channel
            for channel
            in self.channels
            if channel
            in {
                "ct",
                "dose",
                "oral",
                "gtv",
            }
        }

        if (
            "dose_oral"
            in self.channels
        ):
            required_base_channels.update(
                {
                    "dose",
                    "oral",
                }
            )

        if (
            "dose_gtv"
            in self.channels
        ):
            required_base_channels.update(
                {
                    "dose",
                    "gtv",
                }
            )

        arrays = {}

        with np.load(
            npz_path,
            allow_pickle=False,
        ) as data:
            for channel in sorted(
                required_base_channels
            ):
                if channel not in data:
                    raise KeyError(
                        f"病例{patient_id}"
                        f"缺少基础通道{channel}"
                    )

                array = np.asarray(
                    data[
                        channel
                    ],
                    dtype=np.float32,
                )

                if (
                    tuple(
                        array.shape
                    )
                    != EXPECTED_SHAPE_ZYX
                ):
                    raise RuntimeError(
                        f"病例{patient_id}"
                        f"通道{channel}形状错误："
                        f"{array.shape}"
                    )

                if not np.isfinite(
                    array
                ).all():
                    raise RuntimeError(
                        f"病例{patient_id}"
                        f"通道{channel}存在NaN/Inf"
                    )

                arrays[
                    channel
                ] = array

            if (
                "patient_id"
                not in data.files
            ):
                raise KeyError(
                    f"病例{patient_id}的NPZ缺少patient_id"
                )

            internal_patient_id = int(
                np.asarray(data["patient_id"]).reshape(-1)[0]
            )
            if internal_patient_id != patient_id:
                raise RuntimeError(
                    f"NPZ patient_id mismatch: filename={patient_id}, "
                    f"payload={internal_patient_id}"
                )

        if (
            "oral"
            in arrays
        ):
            oral_mask = (
                arrays[
                    "oral"
                ]
                > 0.5
            ).astype(
                np.float32
            )

            if int(
                oral_mask.sum()
            ) <= 0:
                raise RuntimeError(
                    f"病例{patient_id}的oral为空"
                )

            # 向CNN提供严格二值mask。
            arrays[
                "oral"
            ] = oral_mask

        if (
            "gtv"
            in arrays
        ):
            gtv_mask = (
                arrays[
                    "gtv"
                ]
                > 0.5
            ).astype(
                np.float32
            )

            if int(
                gtv_mask.sum()
            ) <= 0:
                raise RuntimeError(
                    f"病例{patient_id}的gtv为空"
                )

            # 向CNN提供严格二值mask。
            arrays[
                "gtv"
            ] = gtv_mask

        if (
            "dose_oral"
            in self.channels
        ):
            arrays[
                "dose_oral"
            ] = (
                arrays[
                    "dose"
                ]
                * arrays[
                    "oral"
                ]
            ).astype(
                np.float32,
                copy=False,
            )

            if not np.isfinite(
                arrays[
                    "dose_oral"
                ]
            ).all():
                raise RuntimeError(
                    f"病例{patient_id}"
                    "的dose_oral存在NaN/Inf"
                )

        if (
            "dose_gtv"
            in self.channels
        ):
            arrays[
                "dose_gtv"
            ] = (
                arrays[
                    "dose"
                ]
                * arrays[
                    "gtv"
                ]
            ).astype(
                np.float32,
                copy=False,
            )

            if not np.isfinite(
                arrays[
                    "dose_gtv"
                ]
            ).all():
                raise RuntimeError(
                    f"病例{patient_id}"
                    "的dose_gtv存在NaN/Inf"
                )

        image = np.stack(
            [
                arrays[
                    channel
                ]
                for channel
                in self.channels
            ],
            axis=0,
        ).astype(
            np.float32,
            copy=False,
        )

        expected_image_shape = (
            len(
                self.channels
            ),
            *EXPECTED_SHAPE_ZYX,
        )

        if (
            tuple(
                image.shape
            )
            != expected_image_shape
        ):
            raise RuntimeError(
                f"病例{patient_id}"
                f"最终输入shape错误："
                f"{image.shape}，"
                f"预期{expected_image_shape}"
            )

        # 当前工程验证不启用增强，
        # 保持模型间唯一差异是输入通道。
        if (
            self.training
            and
            ENABLE_AUGMENTATION
        ):
            pass

        return (
            torch.from_numpy(
                np.ascontiguousarray(
                    image
                )
            ),

            torch.tensor(
                float(
                    label
                ),
                dtype=torch.float32,
            ),

            torch.tensor(
                patient_id,
                dtype=torch.int64,
            ),
        )


# =============================================================================
# 8. 轻量3D ResNet-10
# =============================================================================

def make_group_norm(
    channels: int,
) -> nn.GroupNorm:
    groups = 8

    while (
        groups > 1
        and
        channels % groups != 0
    ):
        groups //= 2

    return nn.GroupNorm(
        num_groups=groups,
        num_channels=channels,
    )


class BasicResidualBlock3D(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        stride: int = 1,
    ):
        super().__init__()

        self.conv1 = nn.Conv3d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False,
        )

        self.norm1 = make_group_norm(
            out_channels
        )

        self.relu = nn.ReLU(
            inplace=True
        )

        self.conv2 = nn.Conv3d(
            out_channels,
            out_channels,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )

        self.norm2 = make_group_norm(
            out_channels
        )

        if (
            stride != 1
            or
            in_channels != out_channels
        ):
            self.shortcut = nn.Sequential(
                nn.Conv3d(
                    in_channels,
                    out_channels,
                    kernel_size=1,
                    stride=stride,
                    bias=False,
                ),

                make_group_norm(
                    out_channels
                ),
            )

        else:
            self.shortcut = nn.Identity()

    def forward(
        self,
        x,
    ):
        identity = self.shortcut(
            x
        )

        x = self.conv1(
            x
        )

        x = self.norm1(
            x
        )

        x = self.relu(
            x
        )

        x = self.conv2(
            x
        )

        x = self.norm2(
            x
        )

        x = x + identity

        x = self.relu(
            x
        )

        return x


class LightweightResNet10_3D(nn.Module):
    def __init__(
        self,
        in_channels: int,
        base_channels: int = 16,
        dropout: float = 0.20,
    ):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv3d(
                in_channels,
                base_channels,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False,
            ),

            make_group_norm(
                base_channels
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.MaxPool3d(
                kernel_size=2,
                stride=2,
            ),
        )

        self.layer1 = BasicResidualBlock3D(
            base_channels,
            base_channels,
            stride=1,
        )

        self.layer2 = BasicResidualBlock3D(
            base_channels,
            base_channels * 2,
            stride=2,
        )

        self.layer3 = BasicResidualBlock3D(
            base_channels * 2,
            base_channels * 4,
            stride=2,
        )

        self.layer4 = BasicResidualBlock3D(
            base_channels * 4,
            base_channels * 8,
            stride=2,
        )

        self.global_pool = nn.AdaptiveAvgPool3d(
            output_size=1
        )

        self.dropout = nn.Dropout(
            p=dropout
        )

        self.classifier = nn.Linear(
            base_channels * 8,
            1,
        )

        self._initialize_weights()

    def _initialize_weights(
        self,
    ):
        for module in self.modules():
            if isinstance(
                module,
                nn.Conv3d,
            ):
                nn.init.kaiming_normal_(
                    module.weight,
                    mode="fan_out",
                    nonlinearity="relu",
                )

            elif isinstance(
                module,
                nn.Linear,
            ):
                nn.init.normal_(
                    module.weight,
                    mean=0.0,
                    std=0.01,
                )

                if module.bias is not None:
                    nn.init.zeros_(
                        module.bias
                    )

    def forward(
        self,
        x,
    ):
        x = self.stem(
            x
        )

        x = self.layer1(
            x
        )

        x = self.layer2(
            x
        )

        x = self.layer3(
            x
        )

        x = self.layer4(
            x
        )

        x = self.global_pool(
            x
        )

        x = torch.flatten(
            x,
            1,
        )

        x = self.dropout(
            x
        )

        return self.classifier(
            x
        ).squeeze(1)


# =============================================================================
# 9. DataLoader
# =============================================================================

def make_loader(
    dataframe: pd.DataFrame,
    channels,
    training: bool,
    seed: int,
) -> DataLoader:
    dataset = NPCNPZDataset(
        dataframe=dataframe,
        channels=channels,
        training=training,
    )

    generator = torch.Generator()

    generator.manual_seed(
        seed
    )

    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=training,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        drop_last=False,
        persistent_workers=False,
        generator=generator,
    )


# =============================================================================
# 10. 训练与评价
# =============================================================================

def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer,
    criterion,
    scaler,
    device: torch.device,
    amp_enabled: bool,
    amp_dtype: torch.dtype,
) -> dict:
    model.train()

    total_loss = 0.0
    total_samples = 0
    skipped_nonfinite_batches = 0

    y_true_all = []
    probabilities_all = []

    for (
        images,
        labels,
        _patient_ids,
    ) in loader:

        images = images.to(
            device,
            non_blocking=True,
        )

        labels = labels.to(
            device,
            non_blocking=True,
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type=device.type,
            dtype=amp_dtype,
            enabled=amp_enabled,
        ):
            logits = model(
                images
            )

            loss = criterion(
                logits,
                labels,
            )

        if not torch.isfinite(
            loss
        ):
            skipped_nonfinite_batches += 1
            continue

        scaler.scale(
            loss
        ).backward()

        scaler.unscale_(
            optimizer
        )

        grad_norm = (
            torch.nn.utils
            .clip_grad_norm_(
                model.parameters(),
                MAX_GRAD_NORM,
            )
        )

        if not torch.isfinite(
            grad_norm
        ):
            skipped_nonfinite_batches += 1

            optimizer.zero_grad(
                set_to_none=True
            )

            scaler.update()

            continue

        scaler.step(
            optimizer
        )

        scaler.update()

        batch_size = int(
            labels.shape[0]
        )

        total_loss += (
            float(
                loss.detach().item()
            )
            * batch_size
        )

        total_samples += batch_size

        # 指标计算强制FP32
        probabilities = torch.sigmoid(
            logits.detach().float()
        )

        y_true_all.append(
            labels.detach()
            .cpu()
            .numpy()
        )

        probabilities_all.append(
            probabilities
            .cpu()
            .numpy()
        )

    if total_samples <= 0:
        raise RuntimeError(
            "当前epoch没有有效训练样本"
        )

    y_true = np.concatenate(
        y_true_all
    ).astype(int)

    probabilities = np.concatenate(
        probabilities_all
    ).astype(float)

    return {
        "loss": float(
            total_loss
            / total_samples
        ),

        "roc_auc": safe_auc(
            y_true,
            probabilities,
        ),

        "pr_auc": safe_pr_auc(
            y_true,
            probabilities,
        ),

        "skipped_nonfinite_batches": int(
            skipped_nonfinite_batches
        ),
    }


@torch.no_grad()
def evaluate_loader(
    model: nn.Module,
    loader: DataLoader,
    criterion,
    device: torch.device,
    amp_enabled: bool,
    amp_dtype: torch.dtype,
) -> dict:
    model.eval()

    total_loss = 0.0
    total_samples = 0

    patient_ids_all = []
    y_true_all = []
    probabilities_all = []

    for (
        images,
        labels,
        patient_ids,
    ) in loader:

        images = images.to(
            device,
            non_blocking=True,
        )

        labels_device = labels.to(
            device,
            non_blocking=True,
        )

        with torch.autocast(
            device_type=device.type,
            dtype=amp_dtype,
            enabled=amp_enabled,
        ):
            logits = model(
                images
            )

            loss = criterion(
                logits,
                labels_device,
            )

        # 关键修正：离开autocast后转FP32再计算sigmoid
        probabilities = torch.sigmoid(
            logits.float()
        )

        batch_size = int(
            labels.shape[0]
        )

        total_loss += (
            float(
                loss.detach().float().item()
            )
            * batch_size
        )

        total_samples += batch_size

        patient_ids_all.append(
            patient_ids.numpy()
        )

        y_true_all.append(
            labels.numpy()
        )

        probabilities_all.append(
            probabilities
            .cpu()
            .numpy()
        )

    if total_samples <= 0:
        raise RuntimeError(
            "评价DataLoader为空"
        )

    return {
        "loss": float(
            total_loss
            / total_samples
        ),

        "patient_ids": np.concatenate(
            patient_ids_all
        ).astype(int),

        "y_true": np.concatenate(
            y_true_all
        ).astype(int),

        "probabilities": np.concatenate(
            probabilities_all
        ).astype(float),
    }


def make_prediction_dataframe(
    output: dict,
    source_df: pd.DataFrame,
    model_name: str,
    role_name: str,
) -> pd.DataFrame:
    dataframe = pd.DataFrame({
        "patient_id": output[
            "patient_ids"
        ],

        "y_true": output[
            "y_true"
        ],

        "probability": output[
            "probabilities"
        ],

        "prediction_at_0_5_descriptive_only": (
            output[
                "probabilities"
            ] >= DESCRIPTIVE_THRESHOLD
        ).astype(int),
    })

    metadata = source_df[
        [
            "patient_id",
            "model_center",
            "stratum",
        ]
    ].copy()

    dataframe = dataframe.merge(
        metadata,
        on="patient_id",
        how="left",
        validate="one_to_one",
    )

    dataframe[
        "model_name"
    ] = model_name

    dataframe[
        "dataset_role"
    ] = role_name

    dataframe[
        "repeat"
    ] = REPEAT_NUMBER

    dataframe[
        "fold"
    ] = FOLD_NUMBER

    return (
        dataframe
        .sort_values(
            "patient_id"
        )
        .reset_index(drop=True)
    )


# =============================================================================
# 11. Stage A
# =============================================================================

def run_stage_a(
    model_name: str,
    channels,
    inner_train_df: pd.DataFrame,
    inner_validation_df: pd.DataFrame,
    device: torch.device,
    model_dir: Path,
):
    seed_everything(
        GLOBAL_SEED
    )

    model = LightweightResNet10_3D(
        in_channels=len(
            channels
        ),
        base_channels=BASE_CHANNELS,
        dropout=DROPOUT,
    ).to(
        device
    )

    parameter_count = (
        count_trainable_parameters(
            model
        )
    )

    criterion = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate_for_epoch(
            1
        ),
        weight_decay=WEIGHT_DECAY,
    )

    amp_settings = resolve_amp_settings(
        device
    )

    amp_enabled = bool(
        amp_settings[
            "enabled"
        ]
    )

    amp_dtype = amp_settings[
        "dtype"
    ]

    amp_mode = str(
        amp_settings[
            "mode"
        ]
    )

    scaler = torch.amp.GradScaler(
        device.type,
        enabled=bool(
            amp_settings[
                "scaler_enabled"
            ]
        ),
    )

    print(
        f"[{model_name}] AMP mode："
        f"{amp_mode}"
    )

    train_loader = make_loader(
        inner_train_df,
        channels,
        training=True,
        seed=GLOBAL_SEED,
    )

    validation_loader = make_loader(
        inner_validation_df,
        channels,
        training=False,
        seed=GLOBAL_SEED,
    )

    checkpoint_path = (
        model_dir
        / "stage_A_best_model.pt"
    )

    history_records = []
    recent_validation_losses = []

    best_selection_score = math.inf
    best_epoch = None
    best_inner_loss_raw = None
    no_improvement_epochs = 0

    for epoch in range(
        1,
        MAX_EPOCHS + 1,
    ):
        epoch_start = time.time()

        current_lr = (
            learning_rate_for_epoch(
                epoch
            )
        )

        set_optimizer_lr(
            optimizer,
            current_lr,
        )

        train_output = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            criterion=criterion,
            scaler=scaler,
            device=device,
            amp_enabled=amp_enabled,
            amp_dtype=amp_dtype,
        )

        validation_output = evaluate_loader(
            model=model,
            loader=validation_loader,
            criterion=criterion,
            device=device,
            amp_enabled=amp_enabled,
            amp_dtype=amp_dtype,
        )

        validation_metrics = (
            calculate_binary_metrics(
                validation_output[
                    "y_true"
                ],
                validation_output[
                    "probabilities"
                ],
                threshold=DESCRIPTIVE_THRESHOLD,
            )
        )

        recent_validation_losses.append(
            float(
                validation_output[
                    "loss"
                ]
            )
        )

        if (
            len(
                recent_validation_losses
            )
            > SELECTION_SMOOTHING_WINDOW
        ):
            recent_validation_losses.pop(
                0
            )

        if (
            len(
                recent_validation_losses
            )
            ==
            SELECTION_SMOOTHING_WINDOW
        ):
            selection_score = float(
                np.mean(
                    recent_validation_losses
                )
            )

        else:
            selection_score = np.nan

        eligible_for_checkpoint = (
            epoch
            >= MIN_CHECKPOINT_EPOCH
            and
            np.isfinite(
                selection_score
            )
        )

        improved = (
            eligible_for_checkpoint
            and
            selection_score
            <
            (
                best_selection_score
                - EARLY_STOPPING_MIN_DELTA
            )
        )

        if improved:
            best_selection_score = float(
                selection_score
            )

            best_epoch = int(
                epoch
            )

            best_inner_loss_raw = float(
                validation_output[
                    "loss"
                ]
            )

            no_improvement_epochs = 0

            torch.save(
                {
                    "model_name": (
                        model_name
                    ),

                    "channels": list(
                        channels
                    ),

                    "model_state": (
                        model.state_dict()
                    ),

                    "best_epoch": (
                        best_epoch
                    ),

                    "best_selection_score_smoothed_inner_loss": (
                        best_selection_score
                    ),

                    "best_inner_loss_raw_at_checkpoint": (
                        best_inner_loss_raw
                    ),

                    "parameter_count": (
                        parameter_count
                    ),

                    "training_seed": (
                        GLOBAL_SEED
                    ),

                    "inner_split_seed": (
                        INNER_SPLIT_SEED
                    ),

                    "amp_mode": (
                        amp_mode
                    ),
                },
                checkpoint_path,
            )

        elif eligible_for_checkpoint:
            no_improvement_epochs += 1

        history_records.append({
            "epoch": epoch,

            "learning_rate": (
                current_lr
            ),

            "amp_mode": (
                amp_mode
            ),

            "train_loss": (
                train_output[
                    "loss"
                ]
            ),

            "train_roc_auc_training_mode": (
                train_output[
                    "roc_auc"
                ]
            ),

            "train_pr_auc_training_mode": (
                train_output[
                    "pr_auc"
                ]
            ),

            "inner_validation_loss": (
                validation_output[
                    "loss"
                ]
            ),

            "inner_validation_loss_smoothed": (
                selection_score
            ),

            "inner_validation_roc_auc": (
                validation_metrics[
                    "roc_auc"
                ]
            ),

            "inner_validation_pr_auc": (
                validation_metrics[
                    "pr_auc"
                ]
            ),

            "inner_validation_brier": (
                validation_metrics[
                    "brier"
                ]
            ),

            "inner_probability_min": (
                validation_metrics[
                    "probability_min"
                ]
            ),

            "inner_probability_max": (
                validation_metrics[
                    "probability_max"
                ]
            ),

            "inner_probability_std": (
                validation_metrics[
                    "probability_std"
                ]
            ),

            "checkpoint_eligible": (
                eligible_for_checkpoint
            ),

            "checkpoint_improved": (
                improved
            ),

            "current_best_epoch": (
                best_epoch
            ),

            "no_improvement_epochs": (
                no_improvement_epochs
            ),

            "skipped_nonfinite_batches": (
                train_output[
                    "skipped_nonfinite_batches"
                ]
            ),

            "epoch_seconds": float(
                time.time()
                - epoch_start
            ),
        })

        print(
            f"[{model_name}] "
            f"Stage A | "
            f"Epoch {epoch:03d} | "
            f"LR={current_lr:.6f} | "
            f"TrainLoss="
            f"{train_output['loss']:.4f} | "
            f"InnerLoss="
            f"{validation_output['loss']:.4f} | "
            f"InnerAUC="
            f"{validation_metrics['roc_auc']:.4f} | "
            f"InnerPR="
            f"{validation_metrics['pr_auc']:.4f} | "
            f"BestEpoch={best_epoch} | "
            f"Bad={no_improvement_epochs}/"
            f"{EARLY_STOPPING_PATIENCE} | "
            f"SkippedInf="
            f"{train_output['skipped_nonfinite_batches']}"
        )

        should_stop = (
            epoch
            >= MIN_EARLY_STOPPING_EPOCH
            and
            best_epoch is not None
            and
            no_improvement_epochs
            >= EARLY_STOPPING_PATIENCE
        )

        if should_stop:
            print(
                f"[{model_name}] "
                f"Stage A在epoch {epoch}"
                "触发early stopping。"
            )

            break

    history_df = pd.DataFrame(
        history_records
    )

    history_df.to_csv(
        model_dir
        / "stage_A_history.csv",
        index=False,
        encoding="utf-8-sig",
    )

    if (
        best_epoch is None
        or
        not checkpoint_path.exists()
    ):
        raise RuntimeError(
            f"{model_name}未生成有效Stage A checkpoint"
        )

    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=False,
    )

    model.load_state_dict(
        checkpoint[
            "model_state"
        ]
    )

    best_inner_output = evaluate_loader(
        model=model,
        loader=validation_loader,
        criterion=criterion,
        device=device,
        amp_enabled=amp_enabled,
        amp_dtype=amp_dtype,
    )

    best_inner_metrics = (
        calculate_binary_metrics(
            best_inner_output[
                "y_true"
            ],
            best_inner_output[
                "probabilities"
            ],
            threshold=DESCRIPTIVE_THRESHOLD,
        )
    )

    prediction_df = (
        make_prediction_dataframe(
            output=best_inner_output,
            source_df=inner_validation_df,
            model_name=model_name,
            role_name="inner_validation_20pct",
        )
    )

    prediction_df.to_csv(
        model_dir
        / "inner_validation_predictions_at_best_checkpoint.csv",
        index=False,
        encoding="utf-8-sig",
    )

    return {
        "best_epoch": int(
            best_epoch
        ),

        "best_selection_score": float(
            best_selection_score
        ),

        "best_inner_loss_raw": float(
            best_inner_loss_raw
        ),

        "best_inner_metrics": (
            best_inner_metrics
        ),

        "parameter_count": (
            parameter_count
        ),

        "history_df": (
            history_df
        ),

        "checkpoint_path": (
            checkpoint_path
        ),

        "total_skipped_nonfinite_batches": int(
            history_df[
                "skipped_nonfinite_batches"
            ].sum()
        ),
    }


# =============================================================================
# 12. Stage B
# =============================================================================

def run_stage_b(
    model_name: str,
    channels,
    selected_epoch: int,
    outer_train_df: pd.DataFrame,
    outer_validation_df: pd.DataFrame,
    device: torch.device,
    model_dir: Path,
):
    seed_everything(
        GLOBAL_SEED
    )

    model = LightweightResNet10_3D(
        in_channels=len(
            channels
        ),
        base_channels=BASE_CHANNELS,
        dropout=DROPOUT,
    ).to(
        device
    )

    criterion = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate_for_epoch(
            1
        ),
        weight_decay=WEIGHT_DECAY,
    )

    amp_settings = resolve_amp_settings(
        device
    )

    amp_enabled = bool(
        amp_settings[
            "enabled"
        ]
    )

    amp_dtype = amp_settings[
        "dtype"
    ]

    amp_mode = str(
        amp_settings[
            "mode"
        ]
    )

    scaler = torch.amp.GradScaler(
        device.type,
        enabled=bool(
            amp_settings[
                "scaler_enabled"
            ]
        ),
    )

    print(
        f"[{model_name}] AMP mode："
        f"{amp_mode}"
    )

    train_loader = make_loader(
        outer_train_df,
        channels,
        training=True,
        seed=GLOBAL_SEED,
    )

    train_evaluation_loader = (
        make_loader(
            outer_train_df,
            channels,
            training=False,
            seed=GLOBAL_SEED,
        )
    )

    outer_validation_loader = (
        make_loader(
            outer_validation_df,
            channels,
            training=False,
            seed=GLOBAL_SEED,
        )
    )

    history_records = []

    for epoch in range(
        1,
        selected_epoch + 1,
    ):
        epoch_start = time.time()

        # 关键：始终按照MAX_EPOCHS=120定义的绝对计划取值
        current_lr = (
            learning_rate_for_epoch(
                epoch
            )
        )

        set_optimizer_lr(
            optimizer,
            current_lr,
        )

        train_output = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            criterion=criterion,
            scaler=scaler,
            device=device,
            amp_enabled=amp_enabled,
            amp_dtype=amp_dtype,
        )

        history_records.append({
            "epoch": epoch,

            "learning_rate": (
                current_lr
            ),

            "amp_mode": (
                amp_mode
            ),

            "train_loss": (
                train_output[
                    "loss"
                ]
            ),

            "train_roc_auc_training_mode": (
                train_output[
                    "roc_auc"
                ]
            ),

            "train_pr_auc_training_mode": (
                train_output[
                    "pr_auc"
                ]
            ),

            "skipped_nonfinite_batches": (
                train_output[
                    "skipped_nonfinite_batches"
                ]
            ),

            "epoch_seconds": float(
                time.time()
                - epoch_start
            ),
        })

        print(
            f"[{model_name}] "
            f"Stage B | "
            f"Epoch {epoch:03d}/"
            f"{selected_epoch:03d} | "
            f"LR={current_lr:.6f} | "
            f"TrainLoss="
            f"{train_output['loss']:.4f} | "
            f"TrainAUC="
            f"{train_output['roc_auc']:.4f} | "
            f"TrainPR="
            f"{train_output['pr_auc']:.4f} | "
            f"SkippedInf="
            f"{train_output['skipped_nonfinite_batches']}"
        )

    history_df = pd.DataFrame(
        history_records
    )

    history_df.to_csv(
        model_dir
        / "stage_B_history.csv",
        index=False,
        encoding="utf-8-sig",
    )

    checkpoint_path = (
        model_dir
        / "stage_B_outer_train_final_model.pt"
    )

    torch.save(
        {
            "model_name": model_name,

            "channels": list(
                channels
            ),

            "model_state": (
                model.state_dict()
            ),

            "selected_epoch": int(
                selected_epoch
            ),

            "training_seed": int(
                GLOBAL_SEED
            ),

            "learning_rate_schedule": (
                "120-epoch cosine schedule"
            ),

            "external_validation_used": (
                False
            ),

            "amp_mode": (
                amp_mode
            ),
        },
        checkpoint_path,
    )

    outer_train_output = (
        evaluate_loader(
            model=model,
            loader=train_evaluation_loader,
            criterion=criterion,
            device=device,
            amp_enabled=amp_enabled,
            amp_dtype=amp_dtype,
        )
    )

    outer_validation_output = (
        evaluate_loader(
            model=model,
            loader=outer_validation_loader,
            criterion=criterion,
            device=device,
            amp_enabled=amp_enabled,
            amp_dtype=amp_dtype,
        )
    )

    outer_train_metrics = (
        calculate_binary_metrics(
            outer_train_output[
                "y_true"
            ],
            outer_train_output[
                "probabilities"
            ],
            threshold=DESCRIPTIVE_THRESHOLD,
        )
    )

    outer_validation_metrics = (
        calculate_binary_metrics(
            outer_validation_output[
                "y_true"
            ],
            outer_validation_output[
                "probabilities"
            ],
            threshold=DESCRIPTIVE_THRESHOLD,
        )
    )

    train_predictions_df = (
        make_prediction_dataframe(
            output=outer_train_output,
            source_df=outer_train_df,
            model_name=model_name,
            role_name="outer_train_final_eval",
        )
    )

    validation_predictions_df = (
        make_prediction_dataframe(
            output=outer_validation_output,
            source_df=outer_validation_df,
            model_name=model_name,
            role_name="outer_validation",
        )
    )

    train_predictions_df.to_csv(
        model_dir
        / "outer_train_final_predictions.csv",
        index=False,
        encoding="utf-8-sig",
    )

    validation_predictions_df.to_csv(
        model_dir
        / "outer_validation_predictions.csv",
        index=False,
        encoding="utf-8-sig",
    )

    center_metrics = {}

    for center in [
        "B",
        "C",
    ]:
        center_df = (
            validation_predictions_df.loc[
                validation_predictions_df[
                    "model_center"
                ] == center
            ]
        )

        if len(center_df) == 0:
            continue

        center_metrics[
            center
        ] = calculate_binary_metrics(
            center_df[
                "y_true"
            ].to_numpy(),
            center_df[
                "probability"
            ].to_numpy(),
            threshold=DESCRIPTIVE_THRESHOLD,
        )

    return {
        "outer_train_metrics": (
            outer_train_metrics
        ),

        "outer_validation_metrics": (
            outer_validation_metrics
        ),

        "outer_validation_center_metrics": (
            center_metrics
        ),

        "history_df": (
            history_df
        ),

        "checkpoint_path": (
            checkpoint_path
        ),

        "amp_mode": (
            amp_mode
        ),

        "total_skipped_nonfinite_batches": int(
            history_df[
                "skipped_nonfinite_batches"
            ].sum()
        ),
    }


# =============================================================================
# 13. 单模型完整运行
# =============================================================================

def save_training_plot(
    model_name: str,
    stage_a_history: pd.DataFrame,
    stage_b_history: pd.DataFrame,
    output_path: Path,
) -> None:
    figure = plt.figure(
        figsize=(10, 6)
    )

    axis = figure.add_subplot(
        1,
        1,
        1,
    )

    axis.plot(
        stage_a_history[
            "epoch"
        ],
        stage_a_history[
            "train_loss"
        ],
        label="Stage A train loss",
    )

    axis.plot(
        stage_a_history[
            "epoch"
        ],
        stage_a_history[
            "inner_validation_loss"
        ],
        label="Stage A inner validation loss",
    )

    axis.plot(
        stage_b_history[
            "epoch"
        ],
        stage_b_history[
            "train_loss"
        ],
        linestyle="--",
        label="Stage B outer-train loss",
    )

    axis.set_xlabel(
        "Epoch"
    )

    axis.set_ylabel(
        "BCE loss"
    )

    axis.set_title(
        model_name
    )

    axis.grid(
        alpha=0.25
    )

    axis.legend()

    figure.tight_layout()

    figure.savefig(
        output_path,
        dpi=160,
        bbox_inches="tight",
    )

    plt.close(
        figure
    )


def run_one_model(
    model_name: str,
    channels,
    dataframes: dict,
    device: torch.device,
) -> dict:
    model_start = time.time()

    model_dir = (
        OUTPUT_ROOT
        / model_name
    )

    model_dir.mkdir(
        parents=True,
        exist_ok=False,
    )

    if device.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    print()
    print("=" * 96)
    print(
        f"开始：{model_name}"
    )
    print(
        f"输入通道：{channels}"
    )
    print("=" * 96)

    stage_a = run_stage_a(
        model_name=model_name,
        channels=channels,
        inner_train_df=dataframes[
            "inner_train"
        ],
        inner_validation_df=dataframes[
            "inner_validation"
        ],
        device=device,
        model_dir=model_dir,
    )

    stage_b = run_stage_b(
        model_name=model_name,
        channels=channels,
        selected_epoch=stage_a[
            "best_epoch"
        ],
        outer_train_df=dataframes[
            "outer_train"
        ],
        outer_validation_df=dataframes[
            "outer_validation"
        ],
        device=device,
        model_dir=model_dir,
    )

    save_training_plot(
        model_name=model_name,
        stage_a_history=stage_a[
            "history_df"
        ],
        stage_b_history=stage_b[
            "history_df"
        ],
        output_path=(
            model_dir
            / "training_history.png"
        ),
    )

    peak_memory_gb = np.nan

    if device.type == "cuda":
        peak_memory_gb = float(
            torch.cuda.max_memory_allocated()
            / (1024 ** 3)
        )

    run_minutes = float(
        (
            time.time()
            - model_start
        )
        / 60.0
    )

    center_b_metrics = (
        stage_b[
            "outer_validation_center_metrics"
        ].get(
            "B",
            {},
        )
    )

    center_c_metrics = (
        stage_b[
            "outer_validation_center_metrics"
        ].get(
            "C",
            {},
        )
    )

    result_payload = {
        "model_name": (
            model_name
        ),

        "model_display_name": (
            MODEL_DISPLAY_NAMES[
                model_name
            ]
        ),

        "model_full_name": (
            MODEL_FULL_NAMES[
                model_name
            ]
        ),

        "repeat": (
            REPEAT_NUMBER
        ),

        "fold": (
            FOLD_NUMBER
        ),

        "training_seed": (
            GLOBAL_SEED
        ),

        "inner_split_seed": (
            INNER_SPLIT_SEED
        ),

        "outer_split_seed": (
            OUTER_SPLIT_SEED
        ),

        "input_channels": list(
            channels
        ),

        "model_purpose": (
            MODEL_PURPOSES[
                model_name
            ]
        ),

        "scalar_branch_used": (
            False
        ),

        "derived_masked_dose_channels": [
            channel
            for channel
            in channels
            if channel
            in DERIVED_MASKED_DOSE_CHANNELS
        ],

        "parameter_count": (
            stage_a[
                "parameter_count"
            ]
        ),

        "selected_epoch": (
            stage_a[
                "best_epoch"
            ]
        ),

        "best_smoothed_inner_loss": (
            stage_a[
                "best_selection_score"
            ]
        ),

        "best_raw_inner_loss": (
            stage_a[
                "best_inner_loss_raw"
            ]
        ),

        "best_inner_metrics": (
            stage_a[
                "best_inner_metrics"
            ]
        ),

        "outer_train_metrics": (
            stage_b[
                "outer_train_metrics"
            ]
        ),

        "outer_validation_metrics": (
            stage_b[
                "outer_validation_metrics"
            ]
        ),

        "outer_validation_center_metrics": (
            stage_b[
                "outer_validation_center_metrics"
            ]
        ),

        "peak_cuda_memory_gb": (
            peak_memory_gb
        ),

        "run_minutes": (
            run_minutes
        ),

        "external_validation_used": (
            False
        ),

        "amp_mode": (
            stage_b[
                "amp_mode"
            ]
        ),

        "stage_A_total_skipped_nonfinite_batches": (
            stage_a[
                "total_skipped_nonfinite_batches"
            ]
        ),

        "stage_B_total_skipped_nonfinite_batches": (
            stage_b[
                "total_skipped_nonfinite_batches"
            ]
        ),

        "status": (
            "PASS"
        ),
    }

    save_json(
        result_payload,
        model_dir
        / "model_result.json",
    )

    summary_row = {
        "repeat": (
            REPEAT_NUMBER
        ),

        "fold": (
            FOLD_NUMBER
        ),

        "model_name": (
            model_name
        ),

        "model_display_name": (
            MODEL_DISPLAY_NAMES[
                model_name
            ]
        ),

        "model_full_name": (
            MODEL_FULL_NAMES[
                model_name
            ]
        ),

        "training_seed": (
            GLOBAL_SEED
        ),

        "inner_split_seed": (
            INNER_SPLIT_SEED
        ),

        "outer_split_seed": (
            OUTER_SPLIT_SEED
        ),

        "input_channels": (
            "+".join(
                channels
            )
        ),

        "model_purpose": (
            MODEL_PURPOSES[
                model_name
            ]
        ),

        "scalar_branch_used": (
            False
        ),

        "parameter_count": (
            stage_a[
                "parameter_count"
            ]
        ),

        "selected_epoch": (
            stage_a[
                "best_epoch"
            ]
        ),

        "inner_roc_auc": (
            stage_a[
                "best_inner_metrics"
            ][
                "roc_auc"
            ]
        ),

        "inner_pr_auc": (
            stage_a[
                "best_inner_metrics"
            ][
                "pr_auc"
            ]
        ),

        "inner_brier": (
            stage_a[
                "best_inner_metrics"
            ][
                "brier"
            ]
        ),

        "outer_train_roc_auc": (
            stage_b[
                "outer_train_metrics"
            ][
                "roc_auc"
            ]
        ),

        "outer_train_pr_auc": (
            stage_b[
                "outer_train_metrics"
            ][
                "pr_auc"
            ]
        ),

        "outer_train_brier": (
            stage_b[
                "outer_train_metrics"
            ][
                "brier"
            ]
        ),

        "outer_validation_roc_auc": (
            stage_b[
                "outer_validation_metrics"
            ][
                "roc_auc"
            ]
        ),

        "outer_validation_pr_auc": (
            stage_b[
                "outer_validation_metrics"
            ][
                "pr_auc"
            ]
        ),

        "outer_validation_brier": (
            stage_b[
                "outer_validation_metrics"
            ][
                "brier"
            ]
        ),

        "outer_validation_probability_std": (
            stage_b[
                "outer_validation_metrics"
            ][
                "probability_std"
            ]
        ),

        "outer_validation_center_B_n": (
            center_b_metrics.get(
                "n",
                np.nan,
            )
        ),

        "outer_validation_center_B_roc_auc": (
            center_b_metrics.get(
                "roc_auc",
                np.nan,
            )
        ),

        "outer_validation_center_C_n": (
            center_c_metrics.get(
                "n",
                np.nan,
            )
        ),

        "outer_validation_center_C_roc_auc": (
            center_c_metrics.get(
                "roc_auc",
                np.nan,
            )
        ),

        "peak_cuda_memory_gb": (
            peak_memory_gb
        ),

        "run_minutes": (
            run_minutes
        ),

        "amp_mode": (
            stage_b[
                "amp_mode"
            ]
        ),

        "stage_A_total_skipped_nonfinite_batches": (
            stage_a[
                "total_skipped_nonfinite_batches"
            ]
        ),

        "stage_B_total_skipped_nonfinite_batches": (
            stage_b[
                "total_skipped_nonfinite_batches"
            ]
        ),

        "status": "PASS",
    }

    print()
    print(
        f"{model_name}完成："
    )

    print(
        "  selected epoch："
        f"{stage_a['best_epoch']}"
    )

    print(
        "  outer-train AUC："
        f"{stage_b['outer_train_metrics']['roc_auc']:.4f}"
    )

    print(
        "  outer-validation AUC："
        f"{stage_b['outer_validation_metrics']['roc_auc']:.4f}"
    )

    print(
        "  outer-validation PR-AUC："
        f"{stage_b['outer_validation_metrics']['pr_auc']:.4f}"
    )

    print(
        "  outer-validation Brier："
        f"{stage_b['outer_validation_metrics']['brier']:.4f}"
    )

    print(
        "  External used：False"
    )

    del stage_a
    del stage_b

    gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()

    return summary_row



# =============================================================================
# 14. 正式4×5运行、断点续跑与OOF汇总
# =============================================================================

def append_formal_log(
    message: str,
) -> None:
    timestamp = datetime.now().isoformat(
        timespec="seconds"
    )

    line = (
        f"[{timestamp}] "
        f"{message}"
    )

    print(
        line
    )

    FORMAL_OUTPUT_ROOT.mkdir(
        parents=True,
        exist_ok=True,
    )

    with FORMAL_RUN_LOG_TXT.open(
        "a",
        encoding="utf-8",
    ) as file:
        file.write(
            line
            + "\n"
        )


def canonical_json_sha256(
    payload: dict,
) -> str:
    text = json.dumps(
        payload,
        ensure_ascii=False,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    )

    return hashlib.sha256(
        text.encode(
            "utf-8"
        )
    ).hexdigest()


def build_formal_signature_payload() -> dict:
    return {
        "output_version": (
            OUTPUT_VERSION
        ),

        "npz_dir": (
            str(
                NPZ_DIR
            )
        ),

        "master_xlsx": (
            str(
                MASTER_XLSX
            )
        ),

        "master_sha256": (
            sha256_file(
                MASTER_XLSX
            )
        ),

        "label_source": (
            "NPC_3DCNN_mucositis_master_frozen_BCdev_Aexternal_v2.xlsx"
        ),

        "split_csv_label_read": (
            False
        ),

        "split_csv_label_compared": (
            False
        ),

        "npz_label_read": (
            False
        ),

        "npz_label_checked": (
            False
        ),

        "split_root": (
            str(
                SPLIT_ROOT
            )
        ),

        "splits_locked_json_sha256": (
            sha256_file(
                SPLITS_LOCKED_JSON
            )
        ),

        "repeats_to_run": (
            REPEATS_TO_RUN
        ),

        "folds_to_run": (
            FOLDS_TO_RUN
        ),

        "models_to_run": (
            MODELS_TO_RUN
        ),

        "model_channels": (
            MODEL_CHANNELS
        ),

        "model_display_names": (
            MODEL_DISPLAY_NAMES
        ),

        "base_channels": (
            BASE_CHANNELS
        ),

        "dropout": (
            DROPOUT
        ),

        "batch_size": (
            BATCH_SIZE
        ),

        "max_epochs": (
            MAX_EPOCHS
        ),

        "min_checkpoint_epoch": (
            MIN_CHECKPOINT_EPOCH
        ),

        "min_early_stopping_epoch": (
            MIN_EARLY_STOPPING_EPOCH
        ),

        "early_stopping_patience": (
            EARLY_STOPPING_PATIENCE
        ),

        "early_stopping_min_delta": (
            EARLY_STOPPING_MIN_DELTA
        ),

        "selection_smoothing_window": (
            SELECTION_SMOOTHING_WINDOW
        ),

        "base_lr": (
            BASE_LR
        ),

        "min_lr": (
            MIN_LR
        ),

        "warmup_epochs": (
            WARMUP_EPOCHS
        ),

        "weight_decay": (
            WEIGHT_DECAY
        ),

        "max_grad_norm": (
            MAX_GRAD_NORM
        ),

        "use_amp": (
            USE_AMP
        ),

        "amp_precision_mode": (
            AMP_PRECISION_MODE
        ),

        "enable_augmentation": (
            ENABLE_AUGMENTATION
        ),

        "descriptive_threshold": (
            DESCRIPTIVE_THRESHOLD
        ),

        "scalar_branch_used": (
            False
        ),

        "external_validation_used": (
            False
        ),
    }


def initialize_or_validate_formal_output() -> dict:
    signature_payload = (
        build_formal_signature_payload()
    )

    run_signature = (
        canonical_json_sha256(
            signature_payload
        )
    )

    if FORMAL_OUTPUT_ROOT.exists():
        if not ALLOW_RESUME:
            raise FileExistsError(
                "正式输出目录已经存在，"
                "但ALLOW_RESUME=False：\n"
                f"{FORMAL_OUTPUT_ROOT}"
            )

        if not FORMAL_RUN_CONFIG_JSON.exists():
            raise RuntimeError(
                "正式输出目录已存在，"
                "但缺少formal_4x5_run_config.json；"
                "为避免混用结果，程序拒绝继续。"
            )

        existing_config = json.loads(
            FORMAL_RUN_CONFIG_JSON.read_text(
                encoding="utf-8"
            )
        )

        existing_signature = str(
            existing_config.get(
                "run_signature",
                "",
            )
        )

        if (
            existing_signature
            != run_signature
        ):
            raise RuntimeError(
                "现有正式结果目录的运行签名"
                "与当前代码配置不一致。\n"
                "不能在同一目录混合不同模型、"
                "路径或训练参数。"
            )

        append_formal_log(
            "检测到一致的正式输出目录，"
            "进入断点续跑模式。"
        )

    else:
        FORMAL_OUTPUT_ROOT.mkdir(
            parents=True,
            exist_ok=False,
        )

        AGGREGATE_DIR.mkdir(
            parents=True,
            exist_ok=False,
        )

        config_payload = {
            "created_at": (
                datetime.now().isoformat(
                    timespec="seconds"
                )
            ),

            "run_signature": (
                run_signature
            ),

            "signature_payload": (
                signature_payload
            ),

            "build_note": (
                BUILD_NOTE
            ),

            "m0_note": (
                "M0 Clinical-DVH is reserved for "
                "a separate traditional modeling pipeline."
            ),

            "splits_locked_json_sha256": (
                sha256_file(
                    SPLITS_LOCKED_JSON
                )
            ),

            "master_xlsx_sha256": (
                sha256_file(
                    MASTER_XLSX
                )
            ),

            "label_source": (
                "NPC_3DCNN_mucositis_master_frozen_BCdev_Aexternal_v2.xlsx"
            ),

            "split_csv_label_read": (
                False
            ),

            "split_csv_label_compared": (
                False
            ),

            "npz_label_read": (
                False
            ),

            "npz_label_checked": (
                False
            ),
        }

        save_json(
            config_payload,
            FORMAL_RUN_CONFIG_JSON,
        )

        append_formal_log(
            "已创建正式4×5输出目录和运行配置。"
        )

    AGGREGATE_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    return {
        "run_signature": (
            run_signature
        ),

        "signature_payload": (
            signature_payload
        ),
    }


def get_fold_input_paths(
    repeat_number: int,
    fold_number: int,
) -> dict:
    fold_dir = (
        SPLIT_ROOT
        / f"repeat_{repeat_number:02d}"
        / f"fold_{fold_number:02d}"
    )

    return {
        "fold_dir": (
            fold_dir
        ),

        "outer_train_csv": (
            fold_dir
            / "outer_train.csv"
        ),

        "outer_validation_csv": (
            fold_dir
            / "outer_validation.csv"
        ),

        "inner_train_csv": (
            fold_dir
            / "inner_train.csv"
        ),

        "inner_validation_csv": (
            fold_dir
            / "inner_validation.csv"
        ),

        "fold_config_json": (
            fold_dir
            / "fold_config.json"
        ),
    }


def get_fold_output_dir(
    repeat_number: int,
    fold_number: int,
) -> Path:
    return (
        FORMAL_OUTPUT_ROOT
        / f"repeat_{repeat_number:02d}"
        / f"fold_{fold_number:02d}"
    )


def get_task_dir(
    repeat_number: int,
    fold_number: int,
    model_name: str,
) -> Path:
    return (
        get_fold_output_dir(
            repeat_number,
            fold_number,
        )
        / model_name
    )


def get_task_required_files(
    task_dir: Path,
) -> list:
    return [
        task_dir
        / "TASK_COMPLETED.json",

        task_dir
        / "model_result.json",

        task_dir
        / "stage_A_history.csv",

        task_dir
        / "stage_A_best_model.pt",

        task_dir
        / "stage_B_history.csv",

        task_dir
        / "stage_B_outer_train_final_model.pt",

        task_dir
        / "outer_validation_predictions.csv",
    ]


def task_is_complete(
    repeat_number: int,
    fold_number: int,
    model_name: str,
) -> bool:
    task_dir = get_task_dir(
        repeat_number,
        fold_number,
        model_name,
    )

    required_files = (
        get_task_required_files(
            task_dir
        )
    )

    if not all(
        path.exists()
        for path
        in required_files
    ):
        return False

    try:
        marker = json.loads(
            (
                task_dir
                / "TASK_COMPLETED.json"
            ).read_text(
                encoding="utf-8"
            )
        )

        result = json.loads(
            (
                task_dir
                / "model_result.json"
            ).read_text(
                encoding="utf-8"
            )
        )

        if str(
            marker.get(
                "status",
                "",
            )
        ) != "PASS":
            return False

        if str(
            result.get(
                "status",
                "",
            )
        ) != "PASS":
            return False

        if int(
            marker.get(
                "repeat",
                -1,
            )
        ) != repeat_number:
            return False

        if int(
            marker.get(
                "fold",
                -1,
            )
        ) != fold_number:
            return False

        if str(
            marker.get(
                "model_name",
                "",
            )
        ) != model_name:
            return False

        prediction_df = pd.read_csv(
            task_dir
            / "outer_validation_predictions.csv",
            encoding="utf-8-sig",
        )

        if len(
            prediction_df
        ) not in {
            61,
            62,
        }:
            return False

        if int(
            prediction_df[
                "patient_id"
            ].nunique()
        ) != len(
            prediction_df
        ):
            return False

        return True

    except Exception:
        return False


def write_task_completed_marker(
    repeat_number: int,
    fold_number: int,
    model_name: str,
) -> None:
    task_dir = get_task_dir(
        repeat_number,
        fold_number,
        model_name,
    )

    critical_paths = {
        "model_result_json": (
            task_dir
            / "model_result.json"
        ),

        "stage_A_history_csv": (
            task_dir
            / "stage_A_history.csv"
        ),

        "stage_B_history_csv": (
            task_dir
            / "stage_B_history.csv"
        ),

        "outer_validation_predictions_csv": (
            task_dir
            / "outer_validation_predictions.csv"
        ),
    }

    payload = {
        "status": (
            "PASS"
        ),

        "completed_at": (
            datetime.now().isoformat(
                timespec="seconds"
            )
        ),

        "repeat": (
            repeat_number
        ),

        "fold": (
            fold_number
        ),

        "model_name": (
            model_name
        ),

        "model_display_name": (
            MODEL_DISPLAY_NAMES[
                model_name
            ]
        ),

        "training_seed": (
            GLOBAL_SEED
        ),

        "critical_file_sha256": {
            name: (
                sha256_file(
                    path
                )
            )
            for name, path
            in critical_paths.items()
        },
    }

    save_json(
        payload,
        task_dir
        / "TASK_COMPLETED.json",
    )


def validate_global_locks() -> dict:
    required_paths = [
        (
            PROJECT_DIR,
            "项目目录",
        ),

        (
            NPZ_DIR,
            "v2 NPZ目录",
        ),

        (
            MASTER_XLSX,
            "正式master Excel",
        ),

        (
            SPLIT_ROOT,
            "正式固定划分目录",
        ),

        (
            SPLITS_LOCKED_JSON,
            "SPLITS_LOCKED.json",
        ),
    ]

    for path, description in required_paths:
        if not path.exists():
            raise FileNotFoundError(
                f"{description}不存在：\n"
                f"{path}"
            )

    split_lock = json.loads(
        SPLITS_LOCKED_JSON.read_text(
            encoding="utf-8"
        )
    )

    if not bool(
        split_lock.get(
            "locked",
            False,
        )
    ):
        raise RuntimeError(
            "固定划分尚未锁定。"
        )

    if str(
        split_lock.get(
            "status",
            "",
        )
    ) != "FINAL_SPLITS_LOCKED":
        raise RuntimeError(
            "SPLITS_LOCKED.json状态错误。"
        )

    expected_values = {
        "final_total": 489,
        "development": 309,
        "external": 180,
        "n_repeats": 4,
        "n_folds_per_repeat": 5,
        "total_outer_folds": 20,
    }

    for key, expected_value in expected_values.items():
        actual_value = int(
            split_lock.get(
                key,
                -1,
            )
        )

        if actual_value != expected_value:
            raise RuntimeError(
                f"锁定文件字段{key}错误："
                f"{actual_value} != {expected_value}"
            )

    if str(
        split_lock.get(
            "external_center",
            "",
        )
    ).strip().upper() != "A":
        raise RuntimeError(
            "SPLITS_LOCKED.json的External中心不是A。"
        )

    if not bool(
        split_lock.get(
            "external_locked",
            False,
        )
    ):
        raise RuntimeError(
            "SPLITS_LOCKED.json未锁定Center A External。"
        )

    master_audit = load_master_label_source()

    split_master_sha256 = str(
        split_lock.get(
            "master_sha256",
            "",
        )
    ).strip()

    if (
        split_master_sha256
        != master_audit[
            "master_sha256"
        ]
    ):
        raise RuntimeError(
            "固定划分与当前Frozen Master不匹配。\n"
            f"split master SHA256：{split_master_sha256}\n"
            f"current master SHA256："
            f"{master_audit['master_sha256']}"
        )

    return {
        "status": "PASS",
        "development": 309,
        "external": 180,
        "total_outer_folds": 20,
        "master_path": str(MASTER_XLSX),
        "master_sha256": master_audit[
            "master_sha256"
        ],
        "master_rows_with_patient_id": master_audit[
            "master_rows_with_patient_id"
        ],
        "master_rows_with_valid_binary_label": master_audit[
            "master_rows_with_valid_binary_label"
        ],
        "master_filename": master_audit[
            "master_filename"
        ],
        "master_total_cases": master_audit[
            "master_total_cases"
        ],
        "master_positive_cases": master_audit[
            "master_positive_cases"
        ],
        "master_negative_cases": master_audit[
            "master_negative_cases"
        ],
        "label_source": "NPC_3DCNN_mucositis_master_frozen_BCdev_Aexternal_v2.xlsx",
        "split_csv_label_read": False,
        "split_csv_label_compared": False,
        "npz_label_read": False,
        "npz_label_checked": False,
        "fixed_split_assignments_regenerated_for_BCdev_Aexternal": True,
        "configured_excluded_case_absent_from_fixed_splits": True,
    }


def load_and_audit_one_fold(
    repeat_number: int,
    fold_number: int,
) -> tuple:
    global GLOBAL_SEED
    global INNER_SPLIT_SEED
    global OUTER_SPLIT_SEED
    global REPEAT_NUMBER
    global FOLD_NUMBER
    global FOLD_DIR
    global OUTPUT_ROOT

    REPEAT_NUMBER = int(
        repeat_number
    )

    FOLD_NUMBER = int(
        fold_number
    )

    paths = get_fold_input_paths(
        repeat_number,
        fold_number,
    )

    FOLD_DIR = paths[
        "fold_dir"
    ]

    OUTPUT_ROOT = get_fold_output_dir(
        repeat_number,
        fold_number,
    )

    for key, path in paths.items():
        if not path.exists():
            raise FileNotFoundError(
                f"repeat_{repeat_number:02d}/"
                f"fold_{fold_number:02d}缺少{key}：\n"
                f"{path}"
            )

    fold_config = json.loads(
        paths[
            "fold_config_json"
        ].read_text(
            encoding="utf-8"
        )
    )

    if int(
        fold_config.get(
            "repeat_number",
            -1,
        )
    ) != repeat_number:
        raise RuntimeError(
            "fold_config中的repeat_number不一致。"
        )

    if int(
        fold_config.get(
            "fold_number",
            -1,
        )
    ) != fold_number:
        raise RuntimeError(
            "fold_config中的fold_number不一致。"
        )

    if not bool(
        fold_config.get(
            "external_locked",
            False,
        )
    ):
        raise RuntimeError(
            "fold_config未锁定External。"
        )

    if str(
        fold_config.get(
            "external_center",
            "",
        )
    ).strip().upper() != "A":
        raise RuntimeError(
            "fold_config的External中心不是A。"
        )

    current_master_sha256 = sha256_file(
        MASTER_XLSX
    )

    fold_master_sha256 = str(
        fold_config.get(
            "master_sha256",
            "",
        )
    ).strip()

    if fold_master_sha256 != current_master_sha256:
        raise RuntimeError(
            "fold_config与当前Frozen Master不匹配。\n"
            f"fold master SHA256：{fold_master_sha256}\n"
            f"current master SHA256：{current_master_sha256}"
        )

    split_lock_payload = json.loads(
        SPLITS_LOCKED_JSON.read_text(
            encoding="utf-8"
        )
    )

    if str(
        fold_config.get(
            "labels_locked_sha256",
            "",
        )
    ).strip() != str(
        split_lock_payload.get(
            "labels_locked_sha256",
            "",
        )
    ).strip():
        raise RuntimeError(
            "fold_config与SPLITS_LOCKED.json中的"
            "LABELS_LOCKED哈希不一致。"
        )

    GLOBAL_SEED = int(
        fold_config[
            "training_seed"
        ]
    )

    INNER_SPLIT_SEED = int(
        fold_config[
            "inner_split_seed"
        ]
    )

    OUTER_SPLIT_SEED = int(
        fold_config[
            "outer_split_seed"
        ]
    )

    outer_train_df = read_split_csv(
        paths[
            "outer_train_csv"
        ],
        "outer_train",
    )

    outer_validation_df = read_split_csv(
        paths[
            "outer_validation_csv"
        ],
        "outer_validation",
    )

    inner_train_df = read_split_csv(
        paths[
            "inner_train_csv"
        ],
        "inner_train",
    )

    inner_validation_df = read_split_csv(
        paths[
            "inner_validation_csv"
        ],
        "inner_validation",
    )

    split_audit = (
        audit_locked_split_relationships(
            outer_train_df=(
                outer_train_df
            ),
            outer_validation_df=(
                outer_validation_df
            ),
            inner_train_df=(
                inner_train_df
            ),
            inner_validation_df=(
                inner_validation_df
            ),
        )
    )

    OUTPUT_ROOT.mkdir(
        parents=True,
        exist_ok=True,
    )

    locked_copy_dir = (
        OUTPUT_ROOT
        / "locked_split_inputs"
    )

    locked_copy_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    for source_path in [
        paths[
            "outer_train_csv"
        ],
        paths[
            "outer_validation_csv"
        ],
        paths[
            "inner_train_csv"
        ],
        paths[
            "inner_validation_csv"
        ],
        paths[
            "fold_config_json"
        ],
        SPLITS_LOCKED_JSON,
        MASTER_XLSX,
    ]:
        destination = (
            locked_copy_dir
            / source_path.name
        )

        if not destination.exists():
            shutil.copy2(
                source_path,
                destination,
            )

    fold_audit_payload = {
        "repeat": (
            repeat_number
        ),

        "fold": (
            fold_number
        ),

        "training_seed": (
            GLOBAL_SEED
        ),

        "inner_split_seed": (
            INNER_SPLIT_SEED
        ),

        "outer_split_seed": (
            OUTER_SPLIT_SEED
        ),

        "split_audit": (
            split_audit
        ),

        "external_loaded": (
            False
        ),

        "external_used": (
            False
        ),

        "input_hashes": {
            "outer_train_csv": (
                sha256_file(
                    paths[
                        "outer_train_csv"
                    ]
                )
            ),

            "outer_validation_csv": (
                sha256_file(
                    paths[
                        "outer_validation_csv"
                    ]
                )
            ),

            "inner_train_csv": (
                sha256_file(
                    paths[
                        "inner_train_csv"
                    ]
                )
            ),

            "inner_validation_csv": (
                sha256_file(
                    paths[
                        "inner_validation_csv"
                    ]
                )
            ),

            "fold_config_json": (
                sha256_file(
                    paths[
                        "fold_config_json"
                    ]
                )
            ),
        },
    }

    save_json(
        fold_audit_payload,
        OUTPUT_ROOT
        / "fold_audit.json",
    )

    dataframes = {
        "outer_train": (
            outer_train_df
        ),

        "outer_validation": (
            outer_validation_df
        ),

        "inner_train": (
            inner_train_df
        ),

        "inner_validation": (
            inner_validation_df
        ),
    }

    return (
        dataframes,
        fold_audit_payload,
    )


def summary_row_from_model_result(
    result: dict,
) -> dict:
    center_b_metrics = (
        result.get(
            "outer_validation_center_metrics",
            {},
        ).get(
            "B",
            {},
        )
    )

    center_c_metrics = (
        result.get(
            "outer_validation_center_metrics",
            {},
        ).get(
            "C",
            {},
        )
    )

    inner_metrics = result.get(
        "best_inner_metrics",
        {},
    )

    outer_train_metrics = result.get(
        "outer_train_metrics",
        {},
    )

    outer_validation_metrics = result.get(
        "outer_validation_metrics",
        {},
    )

    return {
        "repeat": (
            int(
                result[
                    "repeat"
                ]
            )
        ),

        "fold": (
            int(
                result[
                    "fold"
                ]
            )
        ),

        "model_name": (
            result[
                "model_name"
            ]
        ),

        "model_display_name": (
            result[
                "model_display_name"
            ]
        ),

        "model_full_name": (
            result[
                "model_full_name"
            ]
        ),

        "training_seed": (
            int(
                result[
                    "training_seed"
                ]
            )
        ),

        "inner_split_seed": (
            int(
                result[
                    "inner_split_seed"
                ]
            )
        ),

        "outer_split_seed": (
            int(
                result[
                    "outer_split_seed"
                ]
            )
        ),

        "input_channels": (
            "+".join(
                result[
                    "input_channels"
                ]
            )
        ),

        "parameter_count": (
            result[
                "parameter_count"
            ]
        ),

        "selected_epoch": (
            result[
                "selected_epoch"
            ]
        ),

        "inner_roc_auc": (
            inner_metrics.get(
                "roc_auc",
                np.nan,
            )
        ),

        "inner_pr_auc": (
            inner_metrics.get(
                "pr_auc",
                np.nan,
            )
        ),

        "inner_brier": (
            inner_metrics.get(
                "brier",
                np.nan,
            )
        ),

        "outer_train_roc_auc": (
            outer_train_metrics.get(
                "roc_auc",
                np.nan,
            )
        ),

        "outer_train_pr_auc": (
            outer_train_metrics.get(
                "pr_auc",
                np.nan,
            )
        ),

        "outer_train_brier": (
            outer_train_metrics.get(
                "brier",
                np.nan,
            )
        ),

        "outer_validation_roc_auc": (
            outer_validation_metrics.get(
                "roc_auc",
                np.nan,
            )
        ),

        "outer_validation_pr_auc": (
            outer_validation_metrics.get(
                "pr_auc",
                np.nan,
            )
        ),

        "outer_validation_brier": (
            outer_validation_metrics.get(
                "brier",
                np.nan,
            )
        ),

        "outer_validation_probability_std": (
            outer_validation_metrics.get(
                "probability_std",
                np.nan,
            )
        ),

        "outer_validation_center_B_n": (
            center_b_metrics.get(
                "n",
                np.nan,
            )
        ),

        "outer_validation_center_B_roc_auc": (
            center_b_metrics.get(
                "roc_auc",
                np.nan,
            )
        ),

        "outer_validation_center_C_n": (
            center_c_metrics.get(
                "n",
                np.nan,
            )
        ),

        "outer_validation_center_C_roc_auc": (
            center_c_metrics.get(
                "roc_auc",
                np.nan,
            )
        ),

        "peak_cuda_memory_gb": (
            result.get(
                "peak_cuda_memory_gb",
                np.nan,
            )
        ),

        "run_minutes": (
            result.get(
                "run_minutes",
                np.nan,
            )
        ),

        "amp_mode": (
            result.get(
                "amp_mode",
                "",
            )
        ),

        "stage_A_total_skipped_nonfinite_batches": (
            result.get(
                "stage_A_total_skipped_nonfinite_batches",
                np.nan,
            )
        ),

        "stage_B_total_skipped_nonfinite_batches": (
            result.get(
                "stage_B_total_skipped_nonfinite_batches",
                np.nan,
            )
        ),

        "status": (
            result.get(
                "status",
                ""
            )
        ),
    }


def write_progress_table() -> pd.DataFrame:
    rows = []

    for repeat_number in REPEATS_TO_RUN:
        for fold_number in FOLDS_TO_RUN:
            for model_name in MODELS_TO_RUN:
                task_dir = get_task_dir(
                    repeat_number,
                    fold_number,
                    model_name,
                )

                complete = task_is_complete(
                    repeat_number,
                    fold_number,
                    model_name,
                )

                rows.append({
                    "repeat": (
                        repeat_number
                    ),

                    "fold": (
                        fold_number
                    ),

                    "model_name": (
                        model_name
                    ),

                    "model_display_name": (
                        MODEL_DISPLAY_NAMES[
                            model_name
                        ]
                    ),

                    "status": (
                        "PASS"
                        if complete
                        else (
                            "INCOMPLETE"
                            if task_dir.exists()
                            else "PENDING"
                        )
                    ),

                    "task_dir": (
                        str(
                            task_dir
                        )
                    ),
                })

    progress_df = pd.DataFrame(
        rows
    )

    progress_df.to_csv(
        FORMAL_PROGRESS_CSV,
        index=False,
        encoding="utf-8-sig",
    )

    return progress_df


def flatten_metric_row(
    model_name: str,
    subset_name: str,
    metrics: dict,
    repeat_number=None,
) -> dict:
    row = {
        "model_name": (
            model_name
        ),

        "model_display_name": (
            MODEL_DISPLAY_NAMES[
                model_name
            ]
        ),

        "model_full_name": (
            MODEL_FULL_NAMES[
                model_name
            ]
        ),

        "subset": (
            subset_name
        ),
    }

    if repeat_number is not None:
        row[
            "repeat"
        ] = int(
            repeat_number
        )

    for key, value in metrics.items():
        row[
            key
        ] = value

    return row


def paired_stratified_bootstrap(
    reference_df: pd.DataFrame,
    candidate_df: pd.DataFrame,
    n_iterations: int,
    seed: int,
) -> dict:
    merge_columns = [
        "patient_id",
        "y_true",
        "model_center",
        "stratum",
    ]

    merged = reference_df[
        merge_columns
        + [
            "oof_probability_mean",
        ]
    ].merge(
        candidate_df[
            merge_columns
            + [
                "oof_probability_mean",
            ]
        ],
        on="patient_id",
        how="inner",
        suffixes=(
            "_reference",
            "_candidate",
        ),
        validate="one_to_one",
    )

    if len(
        merged
    ) != 309:
        raise RuntimeError(
            "配对比较未对齐309例Development。"
        )

    for column in [
        "y_true",
        "model_center",
        "stratum",
    ]:
        left = merged[
            f"{column}_reference"
        ]

        right = merged[
            f"{column}_candidate"
        ]

        if not (
            left.astype(str)
            == right.astype(str)
        ).all():
            raise RuntimeError(
                f"配对比较的{column}不一致。"
            )

    y_true = merged[
        "y_true_reference"
    ].to_numpy(
        dtype=int
    )

    reference_probability = merged[
        "oof_probability_mean_reference"
    ].to_numpy(
        dtype=float
    )

    candidate_probability = merged[
        "oof_probability_mean_candidate"
    ].to_numpy(
        dtype=float
    )

    strata = merged[
        "stratum_reference"
    ].astype(str).to_numpy()

    observed_reference = (
        calculate_binary_metrics(
            y_true,
            reference_probability,
            threshold=(
                DESCRIPTIVE_THRESHOLD
            ),
        )
    )

    observed_candidate = (
        calculate_binary_metrics(
            y_true,
            candidate_probability,
            threshold=(
                DESCRIPTIVE_THRESHOLD
            ),
        )
    )

    observed_deltas = {
        "auc": (
            observed_candidate[
                "roc_auc"
            ]
            - observed_reference[
                "roc_auc"
            ]
        ),

        "pr_auc": (
            observed_candidate[
                "pr_auc"
            ]
            - observed_reference[
                "pr_auc"
            ]
        ),

        "brier": (
            observed_candidate[
                "brier"
            ]
            - observed_reference[
                "brier"
            ]
        ),
    }

    rng = np.random.default_rng(
        seed
    )

    unique_strata = sorted(
        np.unique(
            strata
        ).tolist()
    )

    stratum_indices = {
        stratum: np.flatnonzero(
            strata == stratum
        )
        for stratum
        in unique_strata
    }

    bootstrap_deltas = {
        "auc": [],
        "pr_auc": [],
        "brier": [],
    }

    for _ in range(
        n_iterations
    ):
        sampled_indices = np.concatenate(
            [
                rng.choice(
                    indices,
                    size=len(
                        indices
                    ),
                    replace=True,
                )
                for indices
                in stratum_indices.values()
            ]
        )

        sampled_y = y_true[
            sampled_indices
        ]

        sampled_reference = (
            reference_probability[
                sampled_indices
            ]
        )

        sampled_candidate = (
            candidate_probability[
                sampled_indices
            ]
        )

        reference_metrics = (
            calculate_binary_metrics(
                sampled_y,
                sampled_reference,
                threshold=(
                    DESCRIPTIVE_THRESHOLD
                ),
            )
        )

        candidate_metrics = (
            calculate_binary_metrics(
                sampled_y,
                sampled_candidate,
                threshold=(
                    DESCRIPTIVE_THRESHOLD
                ),
            )
        )

        bootstrap_deltas[
            "auc"
        ].append(
            candidate_metrics[
                "roc_auc"
            ]
            - reference_metrics[
                "roc_auc"
            ]
        )

        bootstrap_deltas[
            "pr_auc"
        ].append(
            candidate_metrics[
                "pr_auc"
            ]
            - reference_metrics[
                "pr_auc"
            ]
        )

        bootstrap_deltas[
            "brier"
        ].append(
            candidate_metrics[
                "brier"
            ]
            - reference_metrics[
                "brier"
            ]
        )

    result = {
        "n_patients": (
            len(
                merged
            )
        ),

        "bootstrap_iterations": (
            n_iterations
        ),

        "reference_auc": (
            observed_reference[
                "roc_auc"
            ]
        ),

        "candidate_auc": (
            observed_candidate[
                "roc_auc"
            ]
        ),

        "reference_pr_auc": (
            observed_reference[
                "pr_auc"
            ]
        ),

        "candidate_pr_auc": (
            observed_candidate[
                "pr_auc"
            ]
        ),

        "reference_brier": (
            observed_reference[
                "brier"
            ]
        ),

        "candidate_brier": (
            observed_candidate[
                "brier"
            ]
        ),
    }

    for metric_name, values in (
        bootstrap_deltas.items()
    ):
        values = np.asarray(
            values,
            dtype=float,
        )

        observed_delta = (
            observed_deltas[
                metric_name
            ]
        )

        lower = float(
            np.nanpercentile(
                values,
                2.5,
            )
        )

        upper = float(
            np.nanpercentile(
                values,
                97.5,
            )
        )

        p_two_sided = float(
            min(
                1.0,
                2.0
                * min(
                    np.mean(
                        values <= 0
                    ),
                    np.mean(
                        values >= 0
                    ),
                ),
            )
        )

        result[
            f"delta_{metric_name}_candidate_minus_reference"
        ] = float(
            observed_delta
        )

        result[
            f"delta_{metric_name}_ci95_lower"
        ] = lower

        result[
            f"delta_{metric_name}_ci95_upper"
        ] = upper

        result[
            f"delta_{metric_name}_bootstrap_p"
        ] = p_two_sided

    return result


def aggregate_formal_results() -> dict:
    AGGREGATE_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    task_summary_rows = []
    prediction_frames = []

    for repeat_number in REPEATS_TO_RUN:
        for fold_number in FOLDS_TO_RUN:
            for model_name in MODELS_TO_RUN:
                if not task_is_complete(
                    repeat_number,
                    fold_number,
                    model_name,
                ):
                    raise RuntimeError(
                        "正式汇总前发现未完成任务："
                        f"repeat={repeat_number}, "
                        f"fold={fold_number}, "
                        f"model={model_name}"
                    )

                task_dir = get_task_dir(
                    repeat_number,
                    fold_number,
                    model_name,
                )

                result = json.loads(
                    (
                        task_dir
                        / "model_result.json"
                    ).read_text(
                        encoding="utf-8"
                    )
                )

                task_summary_rows.append(
                    summary_row_from_model_result(
                        result
                    )
                )

                prediction_df = pd.read_csv(
                    task_dir
                    / "outer_validation_predictions.csv",
                    encoding="utf-8-sig",
                )

                prediction_df[
                    "repeat"
                ] = int(
                    repeat_number
                )

                prediction_df[
                    "fold"
                ] = int(
                    fold_number
                )

                prediction_df[
                    "model_name"
                ] = model_name

                prediction_df[
                    "model_display_name"
                ] = (
                    MODEL_DISPLAY_NAMES[
                        model_name
                    ]
                )

                prediction_frames.append(
                    prediction_df
                )

    task_summary_df = pd.DataFrame(
        task_summary_rows
    ).sort_values(
        [
            "model_name",
            "repeat",
            "fold",
        ]
    ).reset_index(
        drop=True
    )

    task_summary_df.to_csv(
        ALL_TASKS_SUMMARY_CSV,
        index=False,
        encoding="utf-8-sig",
    )

    all_predictions_df = pd.concat(
        prediction_frames,
        ignore_index=True,
    )

    fold_metrics_df = (
        task_summary_df[
            [
                "repeat",
                "fold",
                "model_name",
                "model_display_name",
                "model_full_name",
                "selected_epoch",
                "parameter_count",
                "outer_train_roc_auc",
                "outer_train_pr_auc",
                "outer_train_brier",
                "outer_validation_roc_auc",
                "outer_validation_pr_auc",
                "outer_validation_brier",
                "outer_validation_probability_std",
                "outer_validation_center_B_n",
                "outer_validation_center_B_roc_auc",
                "outer_validation_center_C_n",
                "outer_validation_center_C_roc_auc",
                "run_minutes",
                "amp_mode",
                "stage_A_total_skipped_nonfinite_batches",
                "stage_B_total_skipped_nonfinite_batches",
                "status",
            ]
        ]
        .copy()
    )

    fold_metrics_df.to_csv(
        FOLD_METRICS_CSV,
        index=False,
        encoding="utf-8-sig",
    )

    repeat_metric_rows = []
    patient_metric_rows = []
    patient_average_frames = []
    per_model_long_frames = {}
    per_model_patient_frames = {}
    audit_models = {}

    for model_name in MODELS_TO_RUN:
        model_long_df = (
            all_predictions_df.loc[
                all_predictions_df[
                    "model_name"
                ] == model_name
            ]
            .copy()
            .sort_values(
                [
                    "repeat",
                    "fold",
                    "patient_id",
                ]
            )
            .reset_index(
                drop=True
            )
        )

        if len(
            model_long_df
        ) != (
            309
            * 4
        ):
            raise RuntimeError(
                f"{model_name}的OOF long数量错误："
                f"{len(model_long_df)}"
            )

        if int(
            model_long_df[
                [
                    "repeat",
                    "patient_id",
                ]
            ].drop_duplicates().shape[
                0
            ]
        ) != (
            309
            * 4
        ):
            raise RuntimeError(
                f"{model_name}存在重复或缺失"
                "repeat-patient OOF预测。"
            )

        repeat_counts = (
            model_long_df.groupby(
                "repeat"
            )[
                "patient_id"
            ].nunique()
        )

        expected_repeat_index = set(
            REPEATS_TO_RUN
        )

        if set(
            repeat_counts.index.astype(int)
        ) != expected_repeat_index:
            raise RuntimeError(
                f"{model_name}未覆盖全部4个repeat。"
            )

        if not (
            repeat_counts.astype(int)
            == 309
        ).all():
            raise RuntimeError(
                f"{model_name}某个repeat"
                "未完整覆盖309例。"
            )

        patient_counts = (
            model_long_df.groupby(
                "patient_id"
            ).size()
        )

        if not (
            patient_counts.astype(int)
            == 4
        ).all():
            raise RuntimeError(
                f"{model_name}并非每例恰好4次OOF。"
            )

        for metadata_column in [
            "y_true",
            "model_center",
            "stratum",
        ]:
            metadata_unique_counts = (
                model_long_df.groupby(
                    "patient_id"
                )[
                    metadata_column
                ].nunique()
            )

            if not (
                metadata_unique_counts
                == 1
            ).all():
                raise RuntimeError(
                    f"{model_name}同一病例的"
                    f"{metadata_column}不一致。"
                )

        long_output_path = (
            AGGREGATE_DIR
            / (
                f"{model_name}_"
                "oof_long_1236.csv"
            )
        )

        model_long_df.to_csv(
            long_output_path,
            index=False,
            encoding="utf-8-sig",
        )

        per_model_long_frames[
            model_name
        ] = model_long_df

        for repeat_number in REPEATS_TO_RUN:
            repeat_df = (
                model_long_df.loc[
                    model_long_df[
                        "repeat"
                    ] == repeat_number
                ]
                .copy()
            )

            overall_metrics = (
                calculate_binary_metrics(
                    repeat_df[
                        "y_true"
                    ],
                    repeat_df[
                        "probability"
                    ],
                    threshold=(
                        DESCRIPTIVE_THRESHOLD
                    ),
                )
            )

            repeat_metric_rows.append(
                flatten_metric_row(
                    model_name=(
                        model_name
                    ),
                    subset_name=(
                        "Overall"
                    ),
                    metrics=(
                        overall_metrics
                    ),
                    repeat_number=(
                        repeat_number
                    ),
                )
            )

            for center in [
                "B",
                "C",
            ]:
                center_df = repeat_df.loc[
                    repeat_df[
                        "model_center"
                    ] == center
                ]

                center_metrics = (
                    calculate_binary_metrics(
                        center_df[
                            "y_true"
                        ],
                        center_df[
                            "probability"
                        ],
                        threshold=(
                            DESCRIPTIVE_THRESHOLD
                        ),
                    )
                )

                repeat_metric_rows.append(
                    flatten_metric_row(
                        model_name=(
                            model_name
                        ),
                        subset_name=(
                            f"Center_{center}"
                        ),
                        metrics=(
                            center_metrics
                        ),
                        repeat_number=(
                            repeat_number
                        ),
                    )
                )

        patient_average_df = (
            model_long_df.groupby(
                [
                    "patient_id",
                    "y_true",
                    "model_center",
                    "stratum",
                ],
                as_index=False,
            )
            .agg(
                oof_probability_mean=(
                    "probability",
                    "mean",
                ),

                oof_probability_sd=(
                    "probability",
                    "std",
                ),

                oof_probability_min=(
                    "probability",
                    "min",
                ),

                oof_probability_max=(
                    "probability",
                    "max",
                ),

                oof_prediction_count=(
                    "probability",
                    "size",
                ),
            )
            .sort_values(
                "patient_id"
            )
            .reset_index(
                drop=True
            )
        )

        if len(
            patient_average_df
        ) != 309:
            raise RuntimeError(
                f"{model_name}患者级平均OOF"
                "不是309例。"
            )

        if not (
            patient_average_df[
                "oof_prediction_count"
            ].astype(int)
            == 4
        ).all():
            raise RuntimeError(
                f"{model_name}患者级OOF计数"
                "并非全部为4。"
            )

        patient_average_df[
            "model_name"
        ] = model_name

        patient_average_df[
            "model_display_name"
        ] = (
            MODEL_DISPLAY_NAMES[
                model_name
            ]
        )

        patient_output_path = (
            AGGREGATE_DIR
            / (
                f"{model_name}_"
                "patient_averaged_oof_309.csv"
            )
        )

        patient_average_df.to_csv(
            patient_output_path,
            index=False,
            encoding="utf-8-sig",
        )

        per_model_patient_frames[
            model_name
        ] = patient_average_df

        patient_average_frames.append(
            patient_average_df
        )

        overall_patient_metrics = (
            calculate_binary_metrics(
                patient_average_df[
                    "y_true"
                ],
                patient_average_df[
                    "oof_probability_mean"
                ],
                threshold=(
                    DESCRIPTIVE_THRESHOLD
                ),
            )
        )

        patient_metric_rows.append(
            flatten_metric_row(
                model_name=(
                    model_name
                ),
                subset_name=(
                    "Overall"
                ),
                metrics=(
                    overall_patient_metrics
                ),
            )
        )

        for center in [
            "B",
            "C",
        ]:
            center_df = (
                patient_average_df.loc[
                    patient_average_df[
                        "model_center"
                    ] == center
                ]
            )

            center_metrics = (
                calculate_binary_metrics(
                    center_df[
                        "y_true"
                    ],
                    center_df[
                        "oof_probability_mean"
                    ],
                    threshold=(
                        DESCRIPTIVE_THRESHOLD
                    ),
                )
            )

            patient_metric_rows.append(
                flatten_metric_row(
                    model_name=(
                        model_name
                    ),
                    subset_name=(
                        f"Center_{center}"
                    ),
                    metrics=(
                        center_metrics
                    ),
                )
            )

        audit_models[
            model_name
        ] = {
            "oof_long_n": (
                len(
                    model_long_df
                )
            ),

            "unique_patients": (
                int(
                    model_long_df[
                        "patient_id"
                    ].nunique()
                )
            ),

            "oof_per_patient": (
                sorted(
                    patient_counts.astype(
                        int
                    ).unique().tolist()
                )
            ),

            "repeat_patient_counts": (
                {
                    str(
                        int(
                            repeat_number
                        )
                    ): int(
                        count
                    )
                    for repeat_number, count
                    in repeat_counts.items()
                }
            ),

            "configured_excluded_case_absent": (
                True
            ),

            "status": (
                "PASS"
            ),
        }

    # 检查所有模型患者集合、标签和中心完全一致。
    reference_model = MODELS_TO_RUN[
        0
    ]

    reference_patient_df = (
        per_model_patient_frames[
            reference_model
        ][
            [
                "patient_id",
                "y_true",
                "model_center",
                "stratum",
            ]
        ]
        .sort_values(
            "patient_id"
        )
        .reset_index(
            drop=True
        )
    )

    for model_name in MODELS_TO_RUN[
        1:
    ]:
        candidate_metadata = (
            per_model_patient_frames[
                model_name
            ][
                [
                    "patient_id",
                    "y_true",
                    "model_center",
                    "stratum",
                ]
            ]
            .sort_values(
                "patient_id"
            )
            .reset_index(
                drop=True
            )
        )

        if not reference_patient_df.equals(
            candidate_metadata
        ):
            raise RuntimeError(
                f"{model_name}与{reference_model}"
                "的患者、标签或中心不一致。"
            )

    repeat_metrics_df = pd.DataFrame(
        repeat_metric_rows
    ).sort_values(
        [
            "model_name",
            "repeat",
            "subset",
        ]
    ).reset_index(
        drop=True
    )

    repeat_metrics_df.to_csv(
        REPEAT_METRICS_CSV,
        index=False,
        encoding="utf-8-sig",
    )

    patient_metrics_df = pd.DataFrame(
        patient_metric_rows
    ).sort_values(
        [
            "model_name",
            "subset",
        ]
    ).reset_index(
        drop=True
    )

    patient_metrics_df.to_csv(
        PATIENT_AVERAGED_METRICS_CSV,
        index=False,
        encoding="utf-8-sig",
    )

    model_summary_rows = []

    for model_name in MODELS_TO_RUN:
        model_fold_df = (
            fold_metrics_df.loc[
                fold_metrics_df[
                    "model_name"
                ] == model_name
            ]
        )

        model_repeat_overall_df = (
            repeat_metrics_df.loc[
                (
                    repeat_metrics_df[
                        "model_name"
                    ] == model_name
                )
                &
                (
                    repeat_metrics_df[
                        "subset"
                    ] == "Overall"
                )
            ]
        )

        model_patient_overall = (
            patient_metrics_df.loc[
                (
                    patient_metrics_df[
                        "model_name"
                    ] == model_name
                )
                &
                (
                    patient_metrics_df[
                        "subset"
                    ] == "Overall"
                )
            ]
            .iloc[
                0
            ]
        )

        model_patient_center_b = (
            patient_metrics_df.loc[
                (
                    patient_metrics_df[
                        "model_name"
                    ] == model_name
                )
                &
                (
                    patient_metrics_df[
                        "subset"
                    ] == "Center_B"
                )
            ]
            .iloc[
                0
            ]
        )

        model_patient_center_c = (
            patient_metrics_df.loc[
                (
                    patient_metrics_df[
                        "model_name"
                    ] == model_name
                )
                &
                (
                    patient_metrics_df[
                        "subset"
                    ] == "Center_C"
                )
            ]
            .iloc[
                0
            ]
        )

        model_summary_rows.append({
            "model_name": (
                model_name
            ),

            "model_display_name": (
                MODEL_DISPLAY_NAMES[
                    model_name
                ]
            ),

            "model_full_name": (
                MODEL_FULL_NAMES[
                    model_name
                ]
            ),

            "input_channels": (
                "+".join(
                    MODEL_CHANNELS[
                        model_name
                    ]
                )
            ),

            "n_folds": (
                len(
                    model_fold_df
                )
            ),

            "selected_epoch_mean": (
                float(
                    model_fold_df[
                        "selected_epoch"
                    ].mean()
                )
            ),

            "selected_epoch_sd": (
                float(
                    model_fold_df[
                        "selected_epoch"
                    ].std(
                        ddof=1
                    )
                )
            ),

            "selected_epoch_median": (
                float(
                    model_fold_df[
                        "selected_epoch"
                    ].median()
                )
            ),

            "fold_auc_mean": (
                float(
                    model_fold_df[
                        "outer_validation_roc_auc"
                    ].mean()
                )
            ),

            "fold_auc_sd": (
                float(
                    model_fold_df[
                        "outer_validation_roc_auc"
                    ].std(
                        ddof=1
                    )
                )
            ),

            "fold_pr_auc_mean": (
                float(
                    model_fold_df[
                        "outer_validation_pr_auc"
                    ].mean()
                )
            ),

            "fold_pr_auc_sd": (
                float(
                    model_fold_df[
                        "outer_validation_pr_auc"
                    ].std(
                        ddof=1
                    )
                )
            ),

            "fold_brier_mean": (
                float(
                    model_fold_df[
                        "outer_validation_brier"
                    ].mean()
                )
            ),

            "fold_brier_sd": (
                float(
                    model_fold_df[
                        "outer_validation_brier"
                    ].std(
                        ddof=1
                    )
                )
            ),

            "repeat_auc_mean": (
                float(
                    model_repeat_overall_df[
                        "roc_auc"
                    ].mean()
                )
            ),

            "repeat_auc_sd": (
                float(
                    model_repeat_overall_df[
                        "roc_auc"
                    ].std(
                        ddof=1
                    )
                )
            ),

            "repeat_pr_auc_mean": (
                float(
                    model_repeat_overall_df[
                        "pr_auc"
                    ].mean()
                )
            ),

            "repeat_pr_auc_sd": (
                float(
                    model_repeat_overall_df[
                        "pr_auc"
                    ].std(
                        ddof=1
                    )
                )
            ),

            "repeat_brier_mean": (
                float(
                    model_repeat_overall_df[
                        "brier"
                    ].mean()
                )
            ),

            "repeat_brier_sd": (
                float(
                    model_repeat_overall_df[
                        "brier"
                    ].std(
                        ddof=1
                    )
                )
            ),

            "patient_averaged_oof_auc": (
                float(
                    model_patient_overall[
                        "roc_auc"
                    ]
                )
            ),

            "patient_averaged_oof_pr_auc": (
                float(
                    model_patient_overall[
                        "pr_auc"
                    ]
                )
            ),

            "patient_averaged_oof_brier": (
                float(
                    model_patient_overall[
                        "brier"
                    ]
                )
            ),

            "patient_averaged_center_B_auc": (
                float(
                    model_patient_center_b[
                        "roc_auc"
                    ]
                )
            ),

            "patient_averaged_center_C_auc": (
                float(
                    model_patient_center_c[
                        "roc_auc"
                    ]
                )
            ),

            "total_run_minutes": (
                float(
                    model_fold_df[
                        "run_minutes"
                    ].sum()
                )
            ),

            "all_tasks_pass": (
                bool(
                    (
                        model_fold_df[
                            "status"
                        ] == "PASS"
                    ).all()
                )
            ),
        })

    model_summary_df = pd.DataFrame(
        model_summary_rows
    )

    model_order = {
        model_name: index
        for index, model_name
        in enumerate(
            MODELS_TO_RUN
        )
    }

    model_summary_df[
        "_order"
    ] = model_summary_df[
        "model_name"
    ].map(
        model_order
    )

    model_summary_df = (
        model_summary_df
        .sort_values(
            "_order"
        )
        .drop(
            columns=[
                "_order",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    model_summary_df.to_csv(
        MODEL_SUMMARY_CSV,
        index=False,
        encoding="utf-8-sig",
    )

    paired_rows = []

    for comparison_index, comparison in enumerate(
        PAIRWISE_COMPARISONS
    ):
        reference_model = comparison[
            "reference_model"
        ]

        candidate_model = comparison[
            "candidate_model"
        ]

        bootstrap_result = (
            paired_stratified_bootstrap(
                reference_df=(
                    per_model_patient_frames[
                        reference_model
                    ]
                ),
                candidate_df=(
                    per_model_patient_frames[
                        candidate_model
                    ]
                ),
                n_iterations=(
                    PAIRED_BOOTSTRAP_ITERATIONS
                ),
                seed=(
                    PAIRED_BOOTSTRAP_SEED
                    + comparison_index
                ),
            )
        )

        paired_rows.append({
            "comparison": (
                comparison[
                    "comparison"
                ]
            ),

            "question": (
                comparison[
                    "question"
                ]
            ),

            "reference_model": (
                reference_model
            ),

            "reference_display_name": (
                MODEL_DISPLAY_NAMES[
                    reference_model
                ]
            ),

            "candidate_model": (
                candidate_model
            ),

            "candidate_display_name": (
                MODEL_DISPLAY_NAMES[
                    candidate_model
                ]
            ),

            **bootstrap_result,
        })

    paired_df = pd.DataFrame(
        paired_rows
    )

    paired_df.to_csv(
        PAIRED_COMPARISONS_CSV,
        index=False,
        encoding="utf-8-sig",
    )

    all_patient_average_df = pd.concat(
        patient_average_frames,
        ignore_index=True,
    )

    design_df = pd.DataFrame(
        FULL_MODEL_DESIGN_ROWS
    )

    with pd.ExcelWriter(
        FORMAL_SUMMARY_XLSX,
        engine="openpyxl",
    ) as writer:
        model_summary_df.to_excel(
            writer,
            sheet_name="Model_summary",
            index=False,
        )

        fold_metrics_df.to_excel(
            writer,
            sheet_name="Fold_metrics_20",
            index=False,
        )

        repeat_metrics_df.to_excel(
            writer,
            sheet_name="Repeat_metrics",
            index=False,
        )

        patient_metrics_df.to_excel(
            writer,
            sheet_name="Patient_avg_metrics",
            index=False,
        )

        paired_df.to_excel(
            writer,
            sheet_name="Paired_comparisons",
            index=False,
        )

        task_summary_df.to_excel(
            writer,
            sheet_name="All_tasks",
            index=False,
        )

        design_df.to_excel(
            writer,
            sheet_name="Model_design",
            index=False,
        )

        all_patient_average_df.to_excel(
            writer,
            sheet_name="Patient_avg_OOF",
            index=False,
        )

    aggregate_audit = {
        "created_at": (
            datetime.now().isoformat(
                timespec="seconds"
            )
        ),

        "models": (
            audit_models
        ),

        "expected_task_count": (
            len(
                REPEATS_TO_RUN
            )
            * len(
                FOLDS_TO_RUN
            )
            * len(
                MODELS_TO_RUN
            )
        ),

        "actual_task_count": (
            len(
                task_summary_df
            )
        ),

        "expected_model_oof_long_n": (
            309
            * 4
        ),

        "expected_patient_averaged_n": (
            309
        ),

        "all_models_metadata_aligned": (
            True
        ),

        "configured_excluded_case_absent": (
            True
        ),

        "external_loaded": (
            False
        ),

        "external_used": (
            False
        ),

        "paired_bootstrap_iterations": (
            PAIRED_BOOTSTRAP_ITERATIONS
        ),

        "status": (
            "PASS"
        ),
    }

    save_json(
        aggregate_audit,
        AGGREGATE_AUDIT_JSON,
    )

    return {
        "task_summary_df": (
            task_summary_df
        ),

        "fold_metrics_df": (
            fold_metrics_df
        ),

        "repeat_metrics_df": (
            repeat_metrics_df
        ),

        "patient_metrics_df": (
            patient_metrics_df
        ),

        "model_summary_df": (
            model_summary_df
        ),

        "paired_df": (
            paired_df
        ),

        "aggregate_audit": (
            aggregate_audit
        ),
    }


def main():
    overall_start = time.time()

    print("=" * 112)
    print(
        "NPC 3D CNN正式4次重复×5折训练"
        "＋OOF汇总"
    )
    print("=" * 112)

    print(
        "Python：",
        sys.version.split()[
            0
        ],
    )

    print(
        "PyTorch：",
        torch.__version__,
    )

    print(
        "CUDA可用：",
        torch.cuda.is_available(),
    )

    if (
        REQUIRE_CUDA
        and
        not torch.cuda.is_available()
    ):
        raise RuntimeError(
            "未检测到CUDA GPU。"
        )

    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    if device.type == "cuda":
        print(
            "GPU：",
            torch.cuda.get_device_name(
                0
            ),
        )

    global_audit = (
        validate_global_locks()
    )

    print("-" * 112)
    print(
        "本次读取的Master文件名：",
        global_audit[
            "master_filename"
        ],
    )
    print(
        "本次读取的Master完整路径：",
        global_audit[
            "master_path"
        ],
    )
    print(
        "病例总数：",
        global_audit[
            "master_total_cases"
        ],
    )
    print(
        "阳性病例总数：",
        global_audit[
            "master_positive_cases"
        ],
    )
    print(
        "阴性病例总数：",
        global_audit[
            "master_negative_cases"
        ],
    )
    print("-" * 112)

    run_config = (
        initialize_or_validate_formal_output()
    )

    save_json(
        global_audit,
        FORMAL_OUTPUT_ROOT
        / "global_lock_audit.json",
    )

    append_formal_log(
        "全局锁定审计PASS："
        "Development=B+C=309，External=A=180，"
        "20 folds，已按BCdev_Aexternal_v2标签和中心重新分层并锁定；"
        "标签来源=NPC_3DCNN_mucositis_master_frozen_BCdev_Aexternal_v2.xlsx；"
        f"Master病例总数={global_audit['master_total_cases']}；"
        f"阳性={global_audit['master_positive_cases']}；"
        f"阴性={global_audit['master_negative_cases']}；"
        "split CSV标签读取=False、比较=False；"
        "NPZ标签读取=False、比较=False。"
    )

    append_formal_log(
        "正式模型顺序："
        + ", ".join(
            MODEL_DISPLAY_NAMES[
                model_name
            ]
            for model_name
            in MODELS_TO_RUN
        )
    )

    append_formal_log(
        "External loaded=False；"
        "External used=False。"
    )

    progress_df = (
        write_progress_table()
    )

    completed_before = int(
        (
            progress_df[
                "status"
            ] == "PASS"
        ).sum()
    )

    total_tasks = int(
        len(
            progress_df
        )
    )

    append_formal_log(
        f"任务总数={total_tasks}；"
        f"运行前已完成={completed_before}。"
    )

    for repeat_number in REPEATS_TO_RUN:
        for fold_number in FOLDS_TO_RUN:
            append_formal_log(
                f"准备repeat_{repeat_number:02d}/"
                f"fold_{fold_number:02d}。"
            )

            dataframes, fold_audit = (
                load_and_audit_one_fold(
                    repeat_number,
                    fold_number,
                )
            )

            append_formal_log(
                f"repeat_{repeat_number:02d}/"
                f"fold_{fold_number:02d}"
                "固定划分审计PASS；"
                f"training_seed={GLOBAL_SEED}。"
            )

            fold_summary_rows = []

            for model_name in MODELS_TO_RUN:
                task_dir = get_task_dir(
                    repeat_number,
                    fold_number,
                    model_name,
                )

                if task_is_complete(
                    repeat_number,
                    fold_number,
                    model_name,
                ):
                    append_formal_log(
                        f"跳过已完成任务："
                        f"repeat_{repeat_number:02d}/"
                        f"fold_{fold_number:02d}/"
                        f"{MODEL_DISPLAY_NAMES[model_name]}"
                    )

                    result = json.loads(
                        (
                            task_dir
                            / "model_result.json"
                        ).read_text(
                            encoding="utf-8"
                        )
                    )

                    fold_summary_rows.append(
                        summary_row_from_model_result(
                            result
                        )
                    )

                    continue

                if task_dir.exists():
                    if not RERUN_INCOMPLETE_TASKS:
                        raise RuntimeError(
                            "发现不完整任务目录，"
                            "但RERUN_INCOMPLETE_TASKS=False：\n"
                            f"{task_dir}"
                        )

                    append_formal_log(
                        "删除不完整任务目录后重跑："
                        f"{task_dir}"
                    )

                    shutil.rmtree(
                        task_dir
                    )

                append_formal_log(
                    f"开始任务："
                    f"repeat_{repeat_number:02d}/"
                    f"fold_{fold_number:02d}/"
                    f"{MODEL_DISPLAY_NAMES[model_name]}"
                )

                try:
                    summary_row = run_one_model(
                        model_name=(
                            model_name
                        ),
                        channels=(
                            MODEL_CHANNELS[
                                model_name
                            ]
                        ),
                        dataframes=(
                            dataframes
                        ),
                        device=(
                            device
                        ),
                    )

                    write_task_completed_marker(
                        repeat_number=(
                            repeat_number
                        ),
                        fold_number=(
                            fold_number
                        ),
                        model_name=(
                            model_name
                        ),
                    )

                    fold_summary_rows.append(
                        summary_row
                    )

                    append_formal_log(
                        f"任务PASS："
                        f"repeat_{repeat_number:02d}/"
                        f"fold_{fold_number:02d}/"
                        f"{MODEL_DISPLAY_NAMES[model_name]}；"
                        f"AUC="
                        f"{summary_row['outer_validation_roc_auc']:.4f}"
                    )

                except Exception as exception:
                    failure_payload = {
                        "status": (
                            "FAIL"
                        ),

                        "failed_at": (
                            datetime.now().isoformat(
                                timespec="seconds"
                            )
                        ),

                        "repeat": (
                            repeat_number
                        ),

                        "fold": (
                            fold_number
                        ),

                        "model_name": (
                            model_name
                        ),

                        "model_display_name": (
                            MODEL_DISPLAY_NAMES[
                                model_name
                            ]
                        ),

                        "error_type": (
                            type(
                                exception
                            ).__name__
                        ),

                        "error_message": (
                            str(
                                exception
                            )
                        ),

                        "traceback": (
                            traceback.format_exc()
                        ),
                    }

                    task_dir.mkdir(
                        parents=True,
                        exist_ok=True,
                    )

                    save_json(
                        failure_payload,
                        task_dir
                        / "TASK_FAILED.json",
                    )

                    append_formal_log(
                        f"任务FAIL："
                        f"repeat_{repeat_number:02d}/"
                        f"fold_{fold_number:02d}/"
                        f"{MODEL_DISPLAY_NAMES[model_name]}；"
                        f"{type(exception).__name__}: "
                        f"{exception}"
                    )

                    write_progress_table()

                    if STOP_ON_TASK_ERROR:
                        raise

            fold_summary_df = pd.DataFrame(
                fold_summary_rows
            )

            fold_summary_df.to_csv(
                get_fold_output_dir(
                    repeat_number,
                    fold_number,
                )
                / "fold_models_summary.csv",
                index=False,
                encoding="utf-8-sig",
            )

            write_progress_table()

            append_formal_log(
                f"repeat_{repeat_number:02d}/"
                f"fold_{fold_number:02d}"
                "全部模型处理完成。"
            )

    final_progress_df = (
        write_progress_table()
    )

    if not (
        final_progress_df[
            "status"
        ] == "PASS"
    ).all():
        incomplete_df = (
            final_progress_df.loc[
                final_progress_df[
                    "status"
                ] != "PASS"
            ]
        )

        raise RuntimeError(
            "仍有未完成任务，不能进行正式汇总：\n"
            + incomplete_df.to_string(
                index=False
            )
        )

    append_formal_log(
        "120个model-fold任务全部PASS，"
        "开始正式OOF汇总和配对bootstrap。"
    )

    aggregate_results = (
        aggregate_formal_results()
    )

    total_minutes = float(
        (
            time.time()
            - overall_start
        )
        / 60.0
    )

    completed_payload = {
        "status": (
            "PASS"
        ),

        "completed_at": (
            datetime.now().isoformat(
                timespec="seconds"
            )
        ),

        "run_signature": (
            run_config[
                "run_signature"
            ]
        ),

        "total_tasks": (
            len(
                REPEATS_TO_RUN
            )
            * len(
                FOLDS_TO_RUN
            )
            * len(
                MODELS_TO_RUN
            )
        ),

        "repeats": (
            len(
                REPEATS_TO_RUN
            )
        ),

        "folds_per_repeat": (
            len(
                FOLDS_TO_RUN
            )
        ),

        "models": (
            MODELS_TO_RUN
        ),

        "patient_averaged_oof_n": (
            309
        ),

        "oof_predictions_per_patient": (
            4
        ),

        "external_loaded": (
            False
        ),

        "external_used": (
            False
        ),

        "total_runtime_minutes_this_invocation": (
            total_minutes
        ),

        "aggregate_audit_status": (
            aggregate_results[
                "aggregate_audit"
            ][
                "status"
            ]
        ),

        "formal_summary_xlsx": (
            str(
                FORMAL_SUMMARY_XLSX
            )
        ),

        "model_summary_csv": (
            str(
                MODEL_SUMMARY_CSV
            )
        ),

        "paired_comparisons_csv": (
            str(
                PAIRED_COMPARISONS_CSV
            )
        ),
    }

    save_json(
        completed_payload,
        FORMAL_COMPLETED_JSON,
    )

    append_formal_log(
        "正式4×5训练和OOF汇总全部完成。"
    )

    print()
    print("=" * 112)
    print(
        "正式4×5训练与OOF汇总完成"
    )
    print("=" * 112)

    print(
        aggregate_results[
            "model_summary_df"
        ].to_string(
            index=False
        )
    )

    print()
    print(
        "关键配对比较："
    )

    print(
        aggregate_results[
            "paired_df"
        ].to_string(
            index=False
        )
    )

    print()
    print(
        "核心汇总："
    )

    print(
        FORMAL_SUMMARY_XLSX
    )

    print(
        "患者级平均OOF与配对结果目录："
    )

    print(
        AGGREGATE_DIR
    )

    print()
    print(
        "External loaded：False"
    )

    print(
        "External used：False"
    )

    print()
    print(
        "下一步不要在本Notebook中评价Center A。"
        "先上传aggregate目录和"
        "FORMAL_4X5_COMPLETED.json复核。"
    )


if __name__ == "__main__":
    main()